# Tarea 03 — Autoencoders para reconstrucción en MVTec AD

Un solo modelo ve **cuatro clases juntas** (cable, capsule, screw, transistor). Hay dos arquitecturas, **VAE** y **U-Net autoencoder**, y cada una se entrena **cuatro veces** cambiando solo la pérdida: `l1`, `l2`, `ssim`, `ssim_l1`. Son **8 runs**.

Imágenes **128×128×3** con valores en **[0, 1]**. Código real (PyTorch Lightning + Hydra + WandB). No hay ficheros `.py`: las clases viven en este notebook y los YAML salen de celdas `%%writefile`.

## Reparto

| Persona | Secciones | Responsabilidad |
| --- | --- | --- |
| P1 | 1–8 | Entorno, Hydra, datos, pérdidas, `BaseAutoencoder`, callbacks, bucle de entrenamiento, análisis |
| P2 | 9 | `VAE` (encoder, reparametrización, decoder, KL con `beta`) |
| P3 | 10 | `UNetAE` (skips y cuello de botella de dimensión `z_dim`) |
| Todos | 0, 11, 12 | Contrato, comparación de los 8 runs, conclusiones |

## Contrato de interfaz

`forward(x) -> (recon, z)`

| Tensor | Forma | Rango |
| --- | --- | --- |
| `x`, `recon` | `[B, 3, 128, 128]` | `[0, 1]` (el decoder **termina en sigmoid**) |
| `z` | `[B, z_dim]` | libre; en el VAE es la **muestra** reparametrizada |

`encode(x) -> z` con la misma forma `[B, z_dim]`. En el VAE, `encode` devuelve **`mu`**, no la muestra.

Batch de todos los splits: `(imagen, etiqueta, clase, subclase)`.

- `etiqueta`: `0` buena, `1` defectuosa (`LongTensor` tras el collate).
- `clase`: `cable` / `capsule` / `screw` / `transistor`.
- `subclase`: carpeta del test (`good` o el nombre del defecto). En train y val es siempre `good`.

Ningún hiperparámetro de entrenamiento va hardcodeado en las clases: salen del `cfg` de Hydra (`z_dim`, `loss`, `lr`, `alpha`, `beta`, `base_channels`, `depth`, `name`).

### Métricas e imágenes (nombres exactos)

| Clave | Quién la escribe | Qué es |
| --- | --- | --- |
| `train/loss`, `val/loss` | `BaseAutoencoder` | reconstrucción + `extra_loss`. `val/loss` es el monitor del checkpoint (`min`) |
| `train/recon_loss`, `val/recon_loss` | `BaseAutoencoder` | solo la pérdida de `build_loss` |
| `train/kl`, `val/kl` | VAE, si rellena `self.last_kl` | KL media del batch **sin** multiplicar por `beta`. La U-Net no la registra |
| `test/recon_error` | `test_step` | MSE medio por imagen, promediado en la época |
| `val/reconstructions` | `ValReconstructionCallback` | 16 de val: original vs reconstruida |
| `val/tsne` | `LatentTSNECallback` | t-SNE de `encode()` en val; la época va en el título |
| `test/reconstructions_good_defective` | `TestReconstructionCallback` | 16 de test, buenas y defectuosas |
| `test/error_histograms` | `plot_error_histograms` | clave extra de análisis (no sustituye a las anteriores) |

`train/loss = recon_loss + extra_loss`. En el VAE, `extra_loss` debe ser `beta * KL`.

Nombres de run: `{model}_{loss}_z{z_dim}`, por ejemplo `vae_ssim_l1_z128` y `u-net_l2_z128`. `group` = nombre del modelo. Tags = `[loss, model]`.

## Cómo ejecutar de principio a fin

Requisito: Python 3.10+ (anotaciones `list[str] | None`). El cwd del kernel tiene que ser la carpeta de este notebook.

1. **Restart & Run All** con las cuatro banderas en `False`. Instala dependencias, escribe `conf/`, comprueba Hydra y las pérdidas, y define las clases. No descarga datos, no entrena y no llama a la API de WandB.
2. Descarga MVTec AD (sección 3), exporta `DATA_DIR` y pon `DOWNLOAD_DATA = True`. Ejecuta la celda de verificación de splits.
3. Login de WandB (entorno, secreto de Colab o el prompt). La API key **no** se escribe en el notebook.
4. P2 implementa `VAE` y, con `RUN_TRAINING = True`, lanza `run_all("vae")`.
5. P3 implementa `UNetAE` y lanza `run_all("u-net")`.
6. Con los 8 runs en el proyecto, `RUN_COMPARISON = True` y ejecuta la sección 11.
7. Completad el análisis crítico y las conclusiones.

Opcional, antes del entrenamiento largo: `RUN_SMOKE = True`. Hay una celda **borrable** que entrena un autoencoder diminuto con `fast_dev_run` (si no hay MVTec, usa tensores sintéticos). No cuenta como uno de los 8 runs.

## Checklist de entrega

- [ ] 8 runs en el proyecto `tarea03-autoencoder-mvtec` con los nombres de arriba
- [ ] Métricas con los nombres exactos de la tabla, `on_epoch=True`
- [ ] Imágenes `val/reconstructions`, `val/tsne`, `test/reconstructions_good_defective`
- [ ] Histogramas de error por subclase de defecto subidos a WandB
- [ ] Análisis crítico del VAE (P2) y de la U-Net (P3)
- [ ] Comparación VAE vs U-Net y efecto de cada pérdida (sección 11), con figuras **descargadas de WandB**
- [ ] Conclusiones

## Conflictos de git en notebooks

Un `.ipynb` es JSON: si dos personas editan a la vez, el merge es casi ilegible.

- Una persona edita **su** sección; las demás no tocan esas celdas mientras tanto.
- Para mirar un diff, exportad a script (`jupyter nbconvert --to script`) o usad `nbstripout` para no commitear salidas ni contadores de ejecución.
- No subáis un `.py` al repo. El contrato de la tarea es que el código vive aquí. `conf/` se regenera al ejecutar las celdas `%%writefile`; no hace falta versionarlo a mano.

## Decisiones ya cerradas

No las reabráis sin anotarlo en la sección de quien cambia:

1. **SSIM** con `pytorch-msssim` y `data_range=1.0` (no torchmetrics). Justificación en la sección 4.
2. Hydra instancia clases del notebook vía un módulo sintético `tarea03` en `sys.modules` (`_target_: tarea03.VAE`). No hay `tarea03.py`.
3. `train/kl` es la KL **sin** `beta`. `beta` solo pondera lo que se suma a `train/loss`.
4. `encode()` del VAE devuelve `mu`. `forward` devuelve `(recon, z_muestra)`. El t-SNE usa `encode()`.
5. El 10 % de validación es un único reparto del train bueno **combinado** (semilla 42), no un 10 % por clase.
6. No hay normalización ImageNet: el espacio es `[0, 1]` para que la sigmoid y el SSIM coincidan.
7. `num_workers=0` por defecto.
8. La vista previa del dataset (sección 3) es la única figura local. Reconstrucciones, t-SNE, histogramas y la comparación salen de WandB.
9. `entity: null` en el YAML: WandB usa la cuenta con la que hiciste login. Puedes fijarla con `WANDB_ENTITY`.


In [ ]:
from __future__ import annotations

# Banderas de control. Dejadlas en False en el commit y en un Restart & Run All
# barato. Actívalas a mano solo en la celda que corresponda.
RUN_TRAINING = False
DOWNLOAD_DATA = False
RUN_SMOKE = False
RUN_COMPARISON = False

print(
    f"RUN_TRAINING={RUN_TRAINING}  DOWNLOAD_DATA={DOWNLOAD_DATA}  "
    f"RUN_SMOKE={RUN_SMOKE}  RUN_COMPARISON={RUN_COMPARISON}"
)


## 1. Setup

**Dueño:** P1

**Qué hace:** instala versiones fijadas, importa librerías, fija la semilla 42, detecta el dispositivo, hace login en WandB sin dejar la API key en el notebook y resuelve `DATA_DIR` (local o Drive de Colab, si ya está montado).

**Entradas:** entorno del kernel, opcionalmente `DATA_DIR`, `WANDB_API_KEY` o el secreto de Colab `WANDB_API_KEY`.

**Salidas:** imports compartidos, `DEVICE`, `DATA_DIR`, módulo sintético `tarea03`, sesión de WandB si había clave.

**No debe tocar otra persona:** versiones de los paquetes, la semilla global y el nombre del módulo `tarea03`. P2 y P3 consumen `DEVICE` y `DATA_DIR`; no los redefinen.


In [ ]:
# Pines para que Colab y un entorno local coincidan.
# Si en Colab esta celda sustituye el torch con CUDA del runtime por una rueda CPU,
# reinstala después la rueda CUDA de tu driver y no cambies el resto de pines.
%pip install -q torch==2.4.1 torchvision==0.19.1 pytorch-lightning==2.4.0 hydra-core==1.3.2 omegaconf==2.3.0 wandb==0.18.7 pytorch-msssim==1.0.0 pandas==2.2.3 matplotlib==3.9.2 scikit-learn==1.5.2 Pillow==10.4.0 tqdm==4.66.5


In [ ]:
from __future__ import annotations

import getpass
import os
import sys
import types
from pathlib import Path
from typing import Any, Callable

import hydra
import matplotlib.pyplot as plt
import pandas as pd
import pytorch_lightning as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import wandb
from hydra.core.global_hydra import GlobalHydra
from omegaconf import DictConfig, OmegaConf
from PIL import Image as PILImage
from pytorch_lightning import seed_everything
from pytorch_lightning.callbacks import Callback, ModelCheckpoint
from pytorch_lightning.loggers import WandbLogger
from pytorch_msssim import ssim as ssim_lote
from sklearn.manifold import TSNE
from torch.utils.data import DataLoader, Dataset
from torchvision.utils import make_grid
from tqdm.auto import tqdm

# Semilla única del enunciado. run_experiment vuelve a llamarla con cfg.seed.
seed_everything(42, workers=True)


def detectar_dispositivo() -> torch.device:
    """Elige cuda, luego mps, y si no hay ninguno, cpu.

    El ``Trainer`` mueve el modelo con ``accelerator=auto``. No hace falta
    llamar a ``model.to(DEVICE)`` antes de ``fit``.
    """
    if torch.cuda.is_available():
        return torch.device("cuda")
    backend_mps = getattr(torch.backends, "mps", None)
    if backend_mps is not None and bool(backend_mps.is_available()):
        return torch.device("mps")
    return torch.device("cpu")


def publicar_en_tarea03(objeto: Any, nombre: str | None = None) -> None:
    """Registra una clase o función en el módulo sintético ``tarea03``.

    Hydra resuelve ``_target_`` importando un módulo de verdad. Un notebook no
    es un paquete, así que publicamos los objetos en ``sys.modules['tarea03']``.
    Así ``_target_: tarea03.VAE`` funciona en Jupyter y en Colab sin ficheros
    ``.py``. El ``__module__`` queda en ``tarea03`` para que un checkpoint se
    pueda cargar en el mismo kernel después de ejecutar estas celdas.
    """
    nombre_publico = objeto.__name__ if nombre is None else nombre
    objeto.__module__ = "tarea03"
    setattr(TAREA03, nombre_publico, objeto)


def registrar_modulo_tarea03() -> types.ModuleType:
    """Crea el módulo ``tarea03`` una sola vez por kernel."""
    existente = sys.modules.get("tarea03")
    if isinstance(existente, types.ModuleType) and existente.__dict__.get("__tarea03_notebook__"):
        return existente
    modulo = types.ModuleType("tarea03")
    modulo.__file__ = "<notebook tarea03_autoencoder_mvtec.ipynb>"
    modulo.__tarea03_notebook__ = True  # type: ignore[attr-defined]
    sys.modules["tarea03"] = modulo
    return modulo


def resolver_data_dir() -> Path:
    """Ruta de MVTec AD.

    Orden: variable ``DATA_DIR``, carpeta ya montada de Google Drive, y por
    último ``./data/mvtec``. No montamos Drive solos: en Colab es opcional.

    .. code-block:: python

        from google.colab import drive
        drive.mount("/content/drive")
        os.environ["DATA_DIR"] = "/content/drive/MyDrive/mvtec_anomaly_detection"
    """
    if os.environ.get("DATA_DIR"):
        return Path(os.environ["DATA_DIR"]).expanduser().resolve()
    raiz_drive = Path("/content/drive/MyDrive")
    if raiz_drive.is_dir():
        for candidato in (raiz_drive / "mvtec_anomaly_detection", raiz_drive / "mvtec_ad", raiz_drive / "mvtec"):
            if candidato.is_dir():
                return candidato.resolve()
    return (Path.cwd() / "data" / "mvtec").resolve()


def iniciar_sesion_wandb() -> None:
    """Login de WandB. La API key no se escribe en el notebook.

    Orden: ``WANDB_API_KEY`` del entorno, secreto de Colab con el mismo nombre,
    y solo si hay un teclado, ``getpass``. En un Run All no interactivo no
    bloquea: el entrenamiento real exigirá la clave más adelante y el smoke
    puede ir en ``WANDB_MODE=offline``.
    """
    clave_entorno = os.environ.get("WANDB_API_KEY", "").strip()
    if clave_entorno:
        wandb.login(key=clave_entorno, relogin=False)
        print("WandB: sesión abierta con WANDB_API_KEY del entorno.")
        return
    try:
        from google.colab import userdata  # type: ignore

        clave_colab = userdata.get("WANDB_API_KEY")
    except Exception:
        clave_colab = None
    if clave_colab:
        os.environ["WANDB_API_KEY"] = str(clave_colab)
        wandb.login(key=str(clave_colab), relogin=False)
        print("WandB: sesión abierta con el secreto de Colab WANDB_API_KEY.")
        return
    if not sys.stdin.isatty():
        print(
            "WandB: no hay API key y el kernel no es interactivo. "
            "Exporta WANDB_API_KEY o crea el secreto en Colab. "
            "El smoke test puede usar WANDB_MODE=offline."
        )
        return
    clave = getpass.getpass("WandB API key (no se guarda en el notebook): ").strip()
    if not clave:
        print("WandB: login omitido.")
        return
    os.environ["WANDB_API_KEY"] = clave
    wandb.login(key=clave)
    print("WandB: sesión abierta.")


TAREA03 = registrar_modulo_tarea03()
DEVICE = detectar_dispositivo()
DATA_DIR = resolver_data_dir()
os.environ["DATA_DIR"] = str(DATA_DIR)
iniciar_sesion_wandb()

print(f"torch {torch.__version__} | lightning {pl.__version__}")
print(f"dispositivo: {DEVICE}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"cwd: {Path.cwd()}")


## 2. Configuración Hydra

**Dueño:** P1

**Qué hace:** escribe `conf/` con celdas `%%writefile` y expone `get_cfg(model, loss)`, que compone un run. Los 8 entrenamientos solo cambian `model` y `model.loss`.

**Entradas:** nada todavía (las clases que Hydra instancia se definen más adelante).

**Salidas:** `conf/config.yaml`, `conf/model/vae.yaml`, `conf/model/u-net.yaml`, `conf/trainer/default.yaml`, `conf/logger/wandb.yaml`, y la función `get_cfg`.

**No debe tocar otra persona:** nombres de grupo (`vae`, `u-net`), claves del cfg y el formato del run `{model}_{loss}_z{z_dim}`. P2 puede leer `beta` y `z_dim`; P3, `base_channels`, `depth` y `z_dim`. No hardcodeéis esos valores en las clases.

### Cómo se resuelve `_target_` desde el notebook

Hydra importa una ruta de módulo (`tarea03.VAE`). Un notebook no es importable, así que la celda de setup mete un módulo sintético en `sys.modules` con el nombre fijo `tarea03` y, cuando cada clase existe, `publicar_en_tarea03` la cuelga ahí.

En esta sección la clase VAE **aún no está definida** (la escribe P2 más abajo). La celda de verificación instancia una sonda `_SondaHydra` publicada en ese módulo y comprueba que `hydra.utils.instantiate` y `hydra.utils.get_class` la encuentran. La sección 9 repite la comprobación con `tarea03.VAE` de verdad.

`get_cfg` llama a `GlobalHydra.instance().clear()` antes de `hydra.initialize(config_path="conf", version_base=None)` para poder relanzar la celda y encadenar los 8 runs. `conf/` se resuelve respecto al directorio del notebook: el cwd del kernel tiene que ser esa carpeta (la misma donde escribe `%%writefile`).


In [ ]:
from __future__ import annotations

from pathlib import Path

for carpeta in (Path("conf/model"), Path("conf/trainer"), Path("conf/logger")):
    carpeta.mkdir(parents=True, exist_ok=True)
print("directorios de conf/ listos en", Path("conf").resolve())


In [ ]:
%%writefile conf/config.yaml
defaults:
  - model: vae
  - trainer: default
  - logger: wandb
  - _self_

seed: 42

data:
  # DATA_DIR lo fija la celda de setup. Si no existe, cae en ./data/mvtec.
  data_dir: ${oc.env:DATA_DIR,./data/mvtec}
  img_size: 128
  batch_size: 32
  # num_workers=0 evita fallos al arrancar DataLoader en Colab y macOS.
  # En Linux, con varios núcleos, se puede subir sin tocar este fichero:
  #   data.num_workers=4
  num_workers: 0
  val_split: 0.1
  classes:
    - cable
    - capsule
    - screw
    - transistor

# El DataModule también se instancia con hydra.utils.instantiate.
datamodule:
  _target_: tarea03.MVTecDataModule
  data_dir: ${data.data_dir}
  img_size: ${data.img_size}
  batch_size: ${data.batch_size}
  num_workers: ${data.num_workers}
  val_split: ${data.val_split}
  classes: ${data.classes}
  seed: ${seed}


In [ ]:
%%writefile conf/model/vae.yaml
# _target_ apunta al módulo sintético `tarea03` del notebook (no hay tarea03.py).
_target_: tarea03.VAE
name: vae
z_dim: 128
loss: l1
lr: 1.0e-3
alpha: 0.5
beta: 1.0


In [ ]:
%%writefile conf/model/u-net.yaml
# Mismo mecanismo que el VAE: la clase se publica en sys.modules['tarea03'].
_target_: tarea03.UNetAE
name: u-net
z_dim: 128
loss: l1
lr: 1.0e-3
alpha: 0.5
base_channels: 32
depth: 4


In [ ]:
%%writefile conf/trainer/default.yaml
max_epochs: 100
accelerator: auto
devices: 1
precision: 32
log_every_n_steps: 10
checkpoint:
  monitor: val/loss
  mode: min
  save_top_k: 1


In [ ]:
%%writefile conf/logger/wandb.yaml
project: tarea03-autoencoder-mvtec
# null = la entidad de la cuenta con la que hiciste login.
# También vale exportar WANDB_ENTITY (tiene prioridad en run_experiment).
entity: null
# Ejemplo: vae_ssim_l1_z128 o u-net_l2_z128.
name: ${model.name}_${model.loss}_z${model.z_dim}
group: ${model.name}
tags:
  - ${model.loss}
  - ${model.name}
log_model: true


In [ ]:
from __future__ import annotations

from pathlib import Path

import hydra
from hydra.core.global_hydra import GlobalHydra
from omegaconf import DictConfig

_MODELOS_CFG = ("vae", "u-net")
_PERDIDAS_CFG = ("l1", "l2", "ssim", "ssim_l1")


def get_cfg(
    model: str,
    loss: str,
    extra_overrides: list[str] | None = None,
) -> DictConfig:
    """Compone la configuración de un run.

    Args:
        model: Grupo Hydra, ``vae`` o ``u-net``.
        loss: ``l1``, ``l2``, ``ssim`` o ``ssim_l1``.
        extra_overrides: Overrides extra. Solo los usa la celda de smoke
            borrable para apuntar ``_target_`` a ``TinyAE``. Los 8 runs reales
            no los pasan.

    Returns:
        ``DictConfig`` listo para ``hydra.utils.instantiate``.
    """
    if model not in _MODELOS_CFG:
        raise ValueError(f"Modelo desconocido: {model!r}. Usa 'vae' o 'u-net'.")
    if loss not in _PERDIDAS_CFG:
        raise ValueError(
            f"Pérdida desconocida: {loss!r}. Opciones: {', '.join(_PERDIDAS_CFG)}."
        )
    if not Path("conf/config.yaml").is_file():
        raise FileNotFoundError(
            "Falta conf/config.yaml. Ejecuta antes las celdas %%writefile "
            "con el cwd en la carpeta del notebook."
        )

    # Sin este clear, la segunda llamada revienta: GlobalHydra already initialized.
    if GlobalHydra.instance().is_initialized():
        GlobalHydra.instance().clear()

    overrides = [f"model={model}", f"model.loss={loss}"]
    if extra_overrides:
        overrides.extend(extra_overrides)

    # Compose API, no @hydra.main: un notebook no tiene un único entrypoint.
    # config_path es relativo al directorio del notebook / al cwd del kernel.
    hydra.initialize(config_path="conf", version_base=None)
    return hydra.compose(config_name="config", overrides=overrides)


publicar_en_tarea03(get_cfg)


In [ ]:
from __future__ import annotations

from typing import Any

import hydra
from omegaconf import OmegaConf


class _SondaHydra:
    """Clase mínima para probar que Hydra importa desde ``tarea03``."""

    def __init__(self, z_dim: int = 8, **kwargs: Any) -> None:
        self.z_dim = z_dim
        self.extra = dict(kwargs)


publicar_en_tarea03(_SondaHydra)

sonda = hydra.utils.instantiate(
    {"_target_": "tarea03._SondaHydra", "z_dim": 16, "name": "sonda"}
)
clase_sonda = hydra.utils.get_class("tarea03._SondaHydra")
assert clase_sonda is _SondaHydra
assert isinstance(sonda, _SondaHydra)
assert sonda.z_dim == 16 and sonda.extra["name"] == "sonda"
print("hydra.utils.instantiate resuelve tarea03._SondaHydra")

cfg_vae = get_cfg("vae", "ssim_l1")
cfg_unet = get_cfg("u-net", "l2")

for bloque in ("data", "model", "datamodule", "trainer", "logger"):
    print(f"# {bloque} (vae / ssim_l1)")
    print(OmegaConf.to_yaml(cfg_vae[bloque], resolve=True))

assert cfg_vae.model._target_ == "tarea03.VAE"
assert cfg_vae.model.loss == "ssim_l1"
assert cfg_vae.model.z_dim == 128
assert abs(float(cfg_vae.model.lr) - 1e-3) < 1e-12
assert float(cfg_vae.model.alpha) == 0.5
assert float(cfg_vae.model.beta) == 1.0
assert cfg_vae.logger.name == "vae_ssim_l1_z128"
assert cfg_vae.logger.group == "vae"
assert list(cfg_vae.logger.tags) == ["ssim_l1", "vae"]
assert cfg_vae.logger.project == "tarea03-autoencoder-mvtec"
assert cfg_vae.data.img_size == 128
assert cfg_vae.data.batch_size == 32
assert cfg_vae.data.num_workers == 0
assert float(cfg_vae.data.val_split) == 0.1
assert list(cfg_vae.data.classes) == ["cable", "capsule", "screw", "transistor"]
assert int(cfg_vae.seed) == 42
assert cfg_vae.datamodule._target_ == "tarea03.MVTecDataModule"
assert cfg_vae.trainer.max_epochs == 100
assert cfg_vae.trainer.checkpoint.monitor == "val/loss"
assert cfg_vae.trainer.checkpoint.mode == "min"
assert int(cfg_vae.trainer.checkpoint.save_top_k) == 1
assert cfg_unet.model._target_ == "tarea03.UNetAE"
assert cfg_unet.logger.name == "u-net_l2_z128"
assert int(cfg_unet.model.base_channels) == 32
assert int(cfg_unet.model.depth) == 4
assert cfg_unet.model.loss == "l2"
print("Hydra compose: ok (VAE real se instancia en la sección 9, cuando exista la clase)")


## 3. Datos

**Dueño:** P1

**Qué hace:** localiza MVTec AD, define `MVTecTestDataset` y `MVTecDataModule`, y deja el reparto train/val reproducible.

**Entradas:** `DATA_DIR` con las cuatro clases. `cfg.data` (`img_size`, `batch_size`, `num_workers`, `val_split`, `classes`, `seed`).

**Salidas:** un `LightningDataModule` que Hydra instancia. Cada ítem es `(imagen, etiqueta, clase, subclase)`.

**No debe tocar otra persona:** el reparto (semilla 42, 10 % de las buenas de train), el collate de strings y el hecho de que train+val solo contienen imágenes **good**. P2 y P3 no leen el disco por su cuenta: usan el datamodule.

### Descarga de MVTec AD

El dataset es de investigación y la descarga oficial pide registro:

<https://www.mvtec.com/company/research/datasets/mvtec-ad>

Descomprime el archivo y apunta `DATA_DIR` a la carpeta que **contiene** las clases (a menudo se llama `mvtec_anomaly_detection`), no a un padre de esa carpeta. Solo hacen falta estas cuatro; el resto puede estar y se ignora.

```text
DATA_DIR/
  cable/
    train/good/*.png
    test/good/*.png
    test/<defecto>/*.png
    ground_truth/...   # no se usa: la etiqueta sale de la carpeta
  capsule/
  screw/
  transistor/
```

En local: `export DATA_DIR=/ruta/a/mvtec_anomaly_detection` antes de lanzar el kernel, o en una celda previa a los imports. En Colab, monta Drive y exporta `DATA_DIR` (snippet en la celda de setup).

**Splits**

- train = imágenes `train/good` de las cuatro clases, menos el 10 % de validación.
- val = ese 10 %. Semilla 42. Decisión: un solo `randperm` sobre el conjunto combinado, no un 10 % por clase.
- test = `test/` oficial, buenas y defectuosas, con clase de objeto y subclase de defecto.

No normalizamos con media y desviación. `ToTensor` deja la imagen en `[0, 1]`, que es el rango de la sigmoid y de `data_range=1.0`.

La celda de la grilla está protegida por `DOWNLOAD_DATA`. Es la única figura que se dibuja en local: solo comprueba que el disco está bien. Las figuras de resultados salen de WandB.


In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Any, Callable

import pytorch_lightning as pl
import torch
import torchvision.transforms as T
from PIL import Image as PILImage
from torch.utils.data import DataLoader, Dataset

EXTENSIONES_IMAGEN = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}


def construir_transform(img_size: int) -> T.Compose:
    """Redimensiona a ``img_size`` y escala a ``[0, 1]`` (sin normalizar)."""
    return T.Compose(
        [
            T.Resize((int(img_size), int(img_size))),
            T.ToTensor(),
        ]
    )


def listar_imagenes(directorio: Path) -> list[Path]:
    """Lista imágenes de una carpeta, ordenadas, para que el split no dependa del filesystem."""
    if not directorio.is_dir():
        return []
    rutas = [
        ruta
        for ruta in directorio.iterdir()
        if ruta.is_file() and ruta.suffix.lower() in EXTENSIONES_IMAGEN
    ]
    return sorted(rutas)


class MVTecTestDataset(Dataset):
    """Test oficial de MVTec AD (buenas y defectuosas).

    Cada elemento es ``(imagen, etiqueta, clase, subclase)``:

    - imagen: ``float32`` ``[3, img_size, img_size]`` en ``[0, 1]``
    - etiqueta: ``0`` si la carpeta es ``good``, ``1`` en caso contrario
    - clase: nombre del objeto
    - subclase: nombre de la carpeta (``good`` o el tipo de defecto)
    """

    def __init__(
        self,
        data_dir: str | Path,
        classes: list[str],
        img_size: int = 128,
        transform: Callable[[PILImage.Image], torch.Tensor] | None = None,
    ) -> None:
        super().__init__()
        self.data_dir = Path(data_dir)
        self.classes = list(classes)
        self.img_size = int(img_size)
        self.transform = transform if transform is not None else construir_transform(self.img_size)
        self.muestras: list[tuple[Path, int, str, str]] = []
        self._indexar()

    def _indexar(self) -> None:
        for clase in self.classes:
            raiz_test = self.data_dir / clase / "test"
            if not raiz_test.is_dir():
                raise FileNotFoundError(
                    f"No existe {raiz_test}. DATA_DIR debe contener "
                    f"{clase}/train/good y {clase}/test/<subclase>."
                )
            subcarpetas = sorted(ruta for ruta in raiz_test.iterdir() if ruta.is_dir())
            for subcarpeta in subcarpetas:
                etiqueta = 0 if subcarpeta.name == "good" else 1
                for ruta in listar_imagenes(subcarpeta):
                    self.muestras.append((ruta, etiqueta, clase, subcarpeta.name))
        if not self.muestras:
            raise RuntimeError(f"El split de test está vacío en {self.data_dir}.")

    def __len__(self) -> int:
        return len(self.muestras)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, int, str, str]:
        ruta, etiqueta, clase, subclase = self.muestras[index]
        with PILImage.open(ruta) as imagen_pil:
            tensor = self.transform(imagen_pil.convert("RGB"))
        return tensor, etiqueta, clase, subclase


class MVTecTrainDataset(Dataset):
    """Buenas de train ya repartidas entre entrenamiento y validación."""

    def __init__(
        self,
        rutas: list[Path],
        clases: list[str],
        img_size: int = 128,
        transform: Callable[[PILImage.Image], torch.Tensor] | None = None,
    ) -> None:
        super().__init__()
        if len(rutas) != len(clases):
            raise ValueError("rutas y clases deben tener la misma longitud.")
        self.rutas = list(rutas)
        self.clases = list(clases)
        self.transform = transform if transform is not None else construir_transform(img_size)

    def __len__(self) -> int:
        return len(self.rutas)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, int, str, str]:
        with PILImage.open(self.rutas[index]) as imagen_pil:
            tensor = self.transform(imagen_pil.convert("RGB"))
        # Train y val salen solo de train/good.
        return tensor, 0, self.clases[index], "good"


def dividir_train_val(
    data_dir: str | Path,
    classes: list[str],
    val_split: float,
    seed: int,
) -> tuple[list[Path], list[str], list[Path], list[str]]:
    """Aparta ``val_split`` de las buenas de train para validación.

    Decisión: un único ``torch.randperm`` con semilla ``seed`` sobre las cuatro
    clases concatenadas (no un 10 % independiente por clase). Es la lectura
    más simple del enunciado. El orden previo al perm es el de ruta ordenada.
    """
    raiz = Path(data_dir)
    rutas: list[Path] = []
    etiquetas_clase: list[str] = []
    for clase in classes:
        carpeta = raiz / clase / "train" / "good"
        imagenes = listar_imagenes(carpeta)
        if not imagenes:
            raise FileNotFoundError(
                f"No hay imágenes en {carpeta}. Descarga MVTec AD y apunta DATA_DIR a la raíz de las clases."
            )
        rutas.extend(imagenes)
        etiquetas_clase.extend([clase] * len(imagenes))

    n_total = len(rutas)
    n_val = int(round(n_total * float(val_split)))
    n_val = min(max(n_val, 1), n_total - 1)
    generador = torch.Generator().manual_seed(int(seed))
    permutacion = torch.randperm(n_total, generator=generador).tolist()
    indices_val = permutacion[:n_val]
    indices_train = permutacion[n_val:]

    def _tomar(indices: list[int]) -> tuple[list[Path], list[str]]:
        return [rutas[i] for i in indices], [etiquetas_clase[i] for i in indices]

    rutas_train, clases_train = _tomar(indices_train)
    rutas_val, clases_val = _tomar(indices_val)
    return rutas_train, clases_train, rutas_val, clases_val


def collate_mvtec(
    batch: list[tuple[torch.Tensor, int, str, str]],
) -> tuple[torch.Tensor, torch.Tensor, list[str], list[str]]:
    """Collate que deja clase y subclase como listas de ``str`` (el default de PyTorch no puede)."""
    imagenes = torch.stack([elemento[0] for elemento in batch], dim=0)
    etiquetas = torch.tensor([elemento[1] for elemento in batch], dtype=torch.long)
    clases = [elemento[2] for elemento in batch]
    subclases = [elemento[3] for elemento in batch]
    return imagenes, etiquetas, clases, subclases


class MVTecDataModule(pl.LightningDataModule):
    """Cuatro clases combinadas, con el split train/val del enunciado.

    - train: ``train/good`` menos ``val_split``
    - val: ese ``val_split`` (solo buenas)
    - test: test oficial, buenas y defectuosas
    """

    def __init__(
        self,
        data_dir: str,
        img_size: int,
        batch_size: int,
        num_workers: int,
        val_split: float,
        classes: list[str],
        seed: int = 42,
    ) -> None:
        super().__init__()
        self.data_dir = str(data_dir)
        self.img_size = int(img_size)
        self.batch_size = int(batch_size)
        self.num_workers = int(num_workers)
        self.val_split = float(val_split)
        self.classes = [str(clase) for clase in classes]
        self.seed = int(seed)
        self.train_set: MVTecTrainDataset | None = None
        self.val_set: MVTecTrainDataset | None = None
        self.test_set: MVTecTestDataset | None = None

    def setup(self, stage: str | None = None) -> None:
        transformacion = construir_transform(self.img_size)
        if stage in (None, "fit", "validate"):
            rutas_train, clases_train, rutas_val, clases_val = dividir_train_val(
                self.data_dir,
                self.classes,
                self.val_split,
                self.seed,
            )
            self.train_set = MVTecTrainDataset(
                rutas_train, clases_train, self.img_size, transformacion
            )
            self.val_set = MVTecTrainDataset(
                rutas_val, clases_val, self.img_size, transformacion
            )
        if stage in (None, "test"):
            self.test_set = MVTecTestDataset(
                self.data_dir, self.classes, self.img_size, transformacion
            )

    def _loader(self, dataset: Dataset, shuffle: bool) -> DataLoader[Any]:
        generador = torch.Generator().manual_seed(self.seed) if shuffle else None
        return DataLoader(
            dataset,
            batch_size=self.batch_size,
            shuffle=shuffle,
            num_workers=self.num_workers,
            collate_fn=collate_mvtec,
            pin_memory=torch.cuda.is_available(),
            generator=generador,
            drop_last=False,
        )

    def train_dataloader(self) -> DataLoader[Any]:
        if self.train_set is None:
            raise RuntimeError("Llama a setup('fit') antes del dataloader de train.")
        return self._loader(self.train_set, shuffle=True)

    def val_dataloader(self) -> DataLoader[Any]:
        if self.val_set is None:
            raise RuntimeError("Llama a setup('fit') antes del dataloader de val.")
        return self._loader(self.val_set, shuffle=False)

    def test_dataloader(self) -> DataLoader[Any]:
        if self.test_set is None:
            raise RuntimeError("Llama a setup('test') antes del dataloader de test.")
        return self._loader(self.test_set, shuffle=False)


publicar_en_tarea03(MVTecTestDataset)
publicar_en_tarea03(MVTecDataModule)
publicar_en_tarea03(collate_mvtec)

cfg_datos = get_cfg("vae", "l1")
dm_hydra = hydra.utils.instantiate(cfg_datos.datamodule)
assert isinstance(dm_hydra, MVTecDataModule)
assert dm_hydra.classes == ["cable", "capsule", "screw", "transistor"]
assert dm_hydra.img_size == 128
assert dm_hydra.batch_size == 32
assert dm_hydra.num_workers == 0
assert dm_hydra.val_split == 0.1
assert dm_hydra.seed == 42
print("Hydra instancia MVTecDataModule sin leer imágenes todavía.")
print("data_dir =", dm_hydra.data_dir)


In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt

if not DOWNLOAD_DATA:
    print(
        "DOWNLOAD_DATA=False: no se leen imágenes. "
        "Cuando MVTec esté en DATA_DIR, pon la bandera a True y relanza esta celda."
    )
else:
    cfg_vista = get_cfg("vae", "l1")
    dm_vista = hydra.utils.instantiate(cfg_vista.datamodule)
    dm_vista.setup()
    if dm_vista.train_set is None or dm_vista.val_set is None or dm_vista.test_set is None:
        raise RuntimeError("setup() no construyó los tres splits.")
    print(
        f"train={len(dm_vista.train_set)}  val={len(dm_vista.val_set)}  "
        f"test={len(dm_vista.test_set)}"
    )
    for nombre, conjunto in (
        ("train", dm_vista.train_set),
        ("val", dm_vista.val_set),
    ):
        conteo: dict[str, int] = {}
        for clase in conjunto.clases:
            conteo[clase] = conteo.get(clase, 0) + 1
        print(f"  {nombre} por clase: {conteo}")
        faltan = [clase for clase in dm_vista.classes if clase not in conteo]
        if faltan:
            # El reparto no es estratificado: con poca suerte una clase puede no caer en val.
            print(f"  aviso: {nombre} no tiene muestras de {faltan}")

    n_buenas = sum(1 for muestra in dm_vista.test_set.muestras if muestra[1] == 0)
    n_malas = len(dm_vista.test_set) - n_buenas
    print(f"  test buenas={n_buenas}  defectuosas={n_malas}")

    n_mostrar = min(8, len(dm_vista.test_set))
    figura, ejes = plt.subplots(2, 4, figsize=(12, 6))
    for eje, indice in zip(ejes.flat, range(8)):
        eje.axis("off")
        if indice >= n_mostrar:
            continue
        imagen, etiqueta, clase, subclase = dm_vista.test_set[indice]
        eje.imshow(imagen.permute(1, 2, 0).numpy())
        eje.set_title(f"{clase}/{subclase}\nlabel={etiqueta}", fontsize=8)
    figura.suptitle(
        "Control local del test (excepción: el resto de figuras del notebook salen de WandB)"
    )
    figura.tight_layout()
    plt.show()


## 4. Pérdidas

**Dueño:** P1

**Qué hace:** implementa `l1`, `l2`, `ssim` = `1 - SSIM` y `ssim_l1` = `alpha * (1 - SSIM) + (1 - alpha) * L1`, más el MSE por imagen sin reducir el batch.

**Entradas:** `name` y `alpha` del cfg. Tensores `[B, 3, H, W]` en `[0, 1]`.

**Salidas:** `build_loss(name, alpha)` devuelve un callable `(recon, x) -> escalar`. `mse_por_imagen(recon, x)` devuelve `[B]`.

**No debe tocar otra persona:** la fórmula y la librería de SSIM. P2 y P3 llaman a `build_loss` a través de `BaseAutoencoder`; no reimplementan SSIM.

### Por qué `pytorch-msssim` y no torchmetrics

Hace falta **una** implementación. `pytorch-msssim` expone `ssim(..., data_range=1.0, size_average=True)` y no ha movido esa función de sitio. `torchmetrics` pesa más y el SSIM ha cambiado de módulo (`functional` / `image`) entre versiones. El MSE por imagen lo calculamos a mano precisamente para **no** reducir el batch, que es lo que haría la métrica por defecto de torchmetrics.

`data_range=1.0` es obligatorio: el default de la librería es 255.


In [ ]:
from __future__ import annotations

from typing import Callable

import torch
import torch.nn.functional as F
from pytorch_msssim import ssim as ssim_lote

PERDIDAS_VALIDAS: tuple[str, ...] = ("l1", "l2", "ssim", "ssim_l1")


def perdida_l1(recon: torch.Tensor, objetivo: torch.Tensor) -> torch.Tensor:
    """L1 media del batch."""
    return F.l1_loss(recon, objetivo, reduction="mean")


def perdida_l2(recon: torch.Tensor, objetivo: torch.Tensor) -> torch.Tensor:
    """MSE medio del batch (la pérdida ``l2`` del enunciado)."""
    return F.mse_loss(recon, objetivo, reduction="mean")


def perdida_ssim(recon: torch.Tensor, objetivo: torch.Tensor) -> torch.Tensor:
    """``1 - SSIM`` con ``data_range=1.0`` (imágenes en ``[0, 1]``)."""
    valor = ssim_lote(recon, objetivo, data_range=1.0, size_average=True)
    return 1.0 - valor


def perdida_ssim_l1(
    recon: torch.Tensor,
    objetivo: torch.Tensor,
    alpha: float,
) -> torch.Tensor:
    """``alpha * (1 - SSIM) + (1 - alpha) * L1``."""
    return float(alpha) * perdida_ssim(recon, objetivo) + (1.0 - float(alpha)) * perdida_l1(
        recon, objetivo
    )


def build_loss(
    name: str,
    alpha: float,
) -> Callable[[torch.Tensor, torch.Tensor], torch.Tensor]:
    """Devuelve la pérdida de reconstrucción ``(recon, x) -> escalar``.

    ``alpha`` solo entra en ``ssim_l1``, pero se valida siempre: el mismo cfg
    se reutiliza al cambiar únicamente el nombre de la pérdida, y un ``alpha``
    fuera de ``[0, 1]`` debe fallar antes del ``fit``.
    """
    if name not in PERDIDAS_VALIDAS:
        raise ValueError(
            f"Pérdida desconocida: {name!r}. Opciones: {', '.join(PERDIDAS_VALIDAS)}."
        )
    alpha_f = float(alpha)
    if not 0.0 <= alpha_f <= 1.0:
        raise ValueError(f"alpha debe estar en [0, 1]; recibido {alpha}.")

    if name == "l1":
        return perdida_l1
    if name == "l2":
        return perdida_l2
    if name == "ssim":
        return perdida_ssim

    def _ssim_l1(recon: torch.Tensor, objetivo: torch.Tensor) -> torch.Tensor:
        return perdida_ssim_l1(recon, objetivo, alpha_f)

    return _ssim_l1


def mse_por_imagen(recon: torch.Tensor, objetivo: torch.Tensor) -> torch.Tensor:
    """MSE medio de canales y píxeles, sin reducir el batch. Forma ``[B]``."""
    if recon.shape != objetivo.shape:
        raise ValueError(
            f"Formas distintas: recon {tuple(recon.shape)} vs objetivo {tuple(objetivo.shape)}."
        )
    dimensiones = tuple(range(1, recon.ndim))
    return torch.mean((recon - objetivo) ** 2, dim=dimensiones)


publicar_en_tarea03(build_loss)
publicar_en_tarea03(mse_por_imagen)


In [ ]:
from __future__ import annotations

import torch


def _probar_perdidas() -> None:
    """Comprueba forma, nombres y la identidad ``ssim_l1 = alpha*(1-SSIM)+(1-alpha)*L1``."""
    torch.manual_seed(0)
    # 128 coincide con el contrato. La ventana de SSIM es 11, así que 32 también valdría.
    original = torch.rand(2, 3, 128, 128)
    copia = original.clone()
    for nombre in PERDIDAS_VALIDAS:
        casi_cero = build_loss(nombre, 0.5)(copia, original)
        assert casi_cero.ndim == 0, nombre
        assert torch.isfinite(casi_cero), nombre
        assert float(casi_cero) < 1e-5, (nombre, float(casi_cero))

    distinta = torch.rand(2, 3, 128, 128)
    alpha = 0.25
    combinada = build_loss("ssim_l1", alpha)(distinta, original)
    esperada = alpha * build_loss("ssim", alpha)(distinta, original) + (
        1.0 - alpha
    ) * build_loss("l1", alpha)(distinta, original)
    assert torch.allclose(combinada, esperada, atol=1e-5)

    error = mse_por_imagen(distinta, original)
    assert error.shape == (2,)
    manual = torch.mean((distinta[0] - original[0]) ** 2)
    assert torch.allclose(error[0], manual)
    assert torch.isfinite(build_loss("l2", 0.0)(distinta, original))
    assert torch.isfinite(build_loss("ssim_l1", 1.0)(distinta, original))

    for llamada in (
        lambda: build_loss("huber", 0.5),
        lambda: build_loss("l1", 1.5),
        lambda: mse_por_imagen(original, original[:, :2]),
    ):
        try:
            llamada()
        except ValueError:
            continue
        raise AssertionError("una entrada inválida no levantó ValueError")

    print(
        "pruebas de pérdidas: ok | ssim_l1(aleatorio) =",
        float(build_loss("ssim_l1", 0.5)(distinta, original)),
    )


_probar_perdidas()


## 5. Modelo base

**Dueño:** P1

**Qué hace:** `BaseAutoencoder`, el `LightningModule` común. Enchufa la pérdida, el optimizador y el logging. No define capas.

**Entradas:** `z_dim`, `loss`, `lr`, `alpha` y los campos extra del cfg (`beta`, `base_channels`, `depth`, `name`) vía `__init__`.

**Salidas:** `training_step` / `validation_step` / `test_step` con los nombres de métrica de la portada. `configure_optimizers` devuelve Adam con `lr` del cfg.

**No debe tocar otra persona:** los nombres de `self.log`, la firma `forward -> (recon, z)` y `extra_loss`. P2 sobreescribe `encode`, `forward` y `extra_loss`. P3 sobreescribe `encode` y `forward` (la U-Net no tiene KL: deja `extra_loss` en 0).

`encode` en la base lanza `NotImplementedError`. En el VAE debe devolver `mu`. El tamaño 128 no se hardcodea aquí: el decoder tiene que reconstruir el `H×W` de la entrada (el cfg de datos fija 128).


In [ ]:
from __future__ import annotations

from typing import Any

import pandas as pd
import pytorch_lightning as pl
import torch


class BaseAutoencoder(pl.LightningModule):
    """Autoencoder de reconstrucción. VAE y U-Net heredan de aquí.

    Contrato
    --------
    - ``forward(x) -> (recon, z)``
    - ``x`` y ``recon`` comparten forma, en ``[0, 1]``; el decoder termina en sigmoid
    - ``z`` es ``[B, z_dim]``
    - ``encode(x) -> z``. En el VAE eso es ``mu``
    - Los hiperparámetros llegan del cfg. ``save_hyperparameters()`` los guarda
    """

    def __init__(
        self,
        z_dim: int,
        loss: str,
        lr: float,
        alpha: float,
        name: str = "ae",
        beta: float | None = None,
        base_channels: int | None = None,
        depth: int | None = None,
        **kwargs: Any,
    ) -> None:
        super().__init__()
        # kwargs absorbe claves raras de Hydra. No se guardan: no son hparams.
        self.save_hyperparameters(ignore=["kwargs"])
        self.funcion_perdida = build_loss(str(loss), float(alpha))
        # La KL cruda del último paso. El VAE la rellena dentro de extra_loss.
        self.last_kl: torch.Tensor | None = None
        self._buffer_test: list[dict[str, Any]] = []

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        """Latente determinista. El VAE devuelve ``mu``; la U-Net, el cuello de botella."""
        raise NotImplementedError("La subclase debe implementar encode().")

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """Devuelve ``(recon, z)``. ``z`` del VAE es la muestra reparametrizada."""
        raise NotImplementedError("La subclase debe implementar forward() -> (recon, z).")

    def extra_loss(
        self,
        recon: torch.Tensor,
        x: torch.Tensor,
        z: torch.Tensor,
    ) -> torch.Tensor:
        """Término que se suma a la reconstrucción. En la base es 0.

        El VAE debe devolver ``beta * KL`` y asignar ``self.last_kl`` a la KL
        media del batch **sin** ``beta``, para que se registren ``train/kl`` y
        ``val/kl``. La U-Net no sobreescribe este método.
        """
        return x.new_zeros(())

    def _imagen(self, batch: Any) -> torch.Tensor:
        if isinstance(batch, torch.Tensor):
            return batch
        if isinstance(batch, (tuple, list)) and batch and isinstance(batch[0], torch.Tensor):
            return batch[0]
        raise TypeError(
            "El batch debe ser un tensor o la tupla (imagen, etiqueta, clase, subclase)."
        )

    def _paso_comun(self, batch: Any, etapa: str) -> torch.Tensor:
        x = self._imagen(batch)
        recon, z = self.forward(x)
        if recon.shape != x.shape:
            raise ValueError(
                f"recon {tuple(recon.shape)} no coincide con x {tuple(x.shape)}. "
                "El decoder debe copiar el tamaño espacial de la entrada y terminar en sigmoid."
            )
        z_dim = int(self.hparams.z_dim)
        if z.ndim != 2 or z.shape[0] != x.shape[0] or int(z.shape[1]) != z_dim:
            raise ValueError(f"z debe ser [B, {z_dim}], recibido {tuple(z.shape)}.")
        perdida_rec = self.funcion_perdida(recon, x)
        # Se anula antes de extra_loss para no arrastrar la KL de un paso anterior.
        self.last_kl = None
        extra = self.extra_loss(recon, x, z)
        total = perdida_rec + extra
        tamano = int(x.size(0))
        # on_step=False para que el nombre quede exactamente train/loss, sin sufijo _epoch.
        self.log(f"{etapa}/loss", total, on_step=False, on_epoch=True, prog_bar=True, batch_size=tamano)
        self.log(
            f"{etapa}/recon_loss",
            perdida_rec,
            on_step=False,
            on_epoch=True,
            batch_size=tamano,
        )
        if self.last_kl is not None:
            self.log(f"{etapa}/kl", self.last_kl, on_step=False, on_epoch=True, batch_size=tamano)
        return total

    def training_step(self, batch: Any, batch_idx: int) -> torch.Tensor:
        return self._paso_comun(batch, "train")

    def validation_step(self, batch: Any, batch_idx: int) -> torch.Tensor:
        return self._paso_comun(batch, "val")

    def on_test_epoch_start(self) -> None:
        self._buffer_test = []

    def test_step(self, batch: Any, batch_idx: int) -> None:
        """Acumula el MSE por imagen junto con z, etiqueta, clase y subclase.

        Registra ``test/recon_error`` (media del batch, agregada en la época
        ponderando por ``batch_size``).
        """
        if not isinstance(batch, (tuple, list)) or len(batch) != 4:
            raise TypeError("test_step espera (imagen, etiqueta, clase, subclase).")
        x, etiqueta, clase, subclase = batch
        recon, z = self.forward(x)
        error = mse_por_imagen(recon, x)
        self.log(
            "test/recon_error",
            error.mean(),
            on_step=False,
            on_epoch=True,
            prog_bar=True,
            batch_size=int(x.size(0)),
        )
        error_cpu = error.detach().cpu()
        z_cpu = z.detach().cpu()
        etiqueta_cpu = etiqueta.detach().cpu()
        for indice in range(int(x.size(0))):
            self._buffer_test.append(
                {
                    "error": float(error_cpu[indice]),
                    "label": int(etiqueta_cpu[indice]),
                    "clase": str(clase[indice]),
                    "subclase": str(subclase[indice]),
                    "z": [float(valor) for valor in z_cpu[indice].tolist()],
                }
            )

    def errores_de_test(self) -> pd.DataFrame:
        """Lo acumulado por ``test_step``, incluido el latente ``z``.

        ``compute_errors`` hace otro pase y no incluye ``z``: es el DataFrame
        pensado para los histogramas.
        """
        return pd.DataFrame(self._buffer_test)

    def configure_optimizers(self) -> torch.optim.Optimizer:
        parametros = [parametro for parametro in self.parameters() if parametro.requires_grad]
        if not parametros:
            raise RuntimeError(
                "El modelo no tiene parámetros entrenables. "
                "P2/P3 deben definir encoder y decoder antes de fit."
            )
        return torch.optim.Adam(parametros, lr=float(self.hparams.lr))


publicar_en_tarea03(BaseAutoencoder)

_base_comprobacion = BaseAutoencoder(z_dim=8, loss="l1", lr=1e-3, alpha=0.5, name="base")
assert int(_base_comprobacion.hparams.z_dim) == 8
assert _base_comprobacion.hparams.loss == "l1"
_cero = _base_comprobacion.extra_loss(
    torch.zeros(1, 3, 8, 8),
    torch.zeros(1, 3, 8, 8),
    torch.zeros(1, 8),
)
assert float(_cero) == 0.0
try:
    _base_comprobacion.encode(torch.zeros(1, 3, 8, 8))
except NotImplementedError:
    print("BaseAutoencoder: hparams desde __init__, extra_loss=0, encode abstracto.")
else:
    raise AssertionError("encode de la base debía lanzar NotImplementedError")


## 6. Callbacks de WandB

**Dueño:** P1

**Qué hace:** sube a WandB las tres figuras obligatorias.

**Entradas:** el `Trainer`, el módulo y los dataloaders de val o test.

**Salidas:** imágenes con las claves `val/reconstructions`, `val/tsne` y `test/reconstructions_good_defective`.

**No debe tocar otra persona:** las claves y los hooks. P2/P3 no reimplementan el pintado; si su `forward` / `encode` cumple el contrato, estas clases funcionan.

Decisiones: el t-SNE llama a `encode()` (en el VAE eso es `mu`, no la muestra). Se salta el sanity check. Con menos de 4 latentes no hay t-SNE (pasa en un `fast_dev_run` minúsculo). El 16 del enunciado es un argumento del callback, no un hiperparámetro de entrenamiento.


In [ ]:
from __future__ import annotations

from typing import Any

import matplotlib.pyplot as plt
import pytorch_lightning as pl
import torch
import wandb
from pytorch_lightning.callbacks import Callback
from pytorch_lightning.loggers import WandbLogger
from sklearn.manifold import TSNE
from torchvision.utils import make_grid

_COLORES_CLASE = {
    "cable": "tab:blue",
    "capsule": "tab:orange",
    "screw": "tab:green",
    "transistor": "tab:red",
}


def rejilla_original_recon(original: torch.Tensor, reconstruccion: torch.Tensor) -> torch.Tensor:
    """Rejilla CHW en ``[0, 1]``: primero las originales, después las reconstrucciones."""
    n = min(int(original.size(0)), int(reconstruccion.size(0)))
    original_cpu = original[:n].detach().cpu().clamp(0, 1)
    recon_cpu = reconstruccion[:n].detach().cpu().clamp(0, 1)
    columnas = min(n, 8)
    return make_grid(torch.cat([original_cpu, recon_cpu], dim=0), nrow=columnas)


def _logger_wandb(trainer: pl.Trainer) -> WandbLogger | None:
    logger = trainer.logger
    if isinstance(logger, WandbLogger):
        return logger
    return None


def _enviar_wandb(trainer: pl.Trainer, clave: str, imagen: Any, titulo: str) -> None:
    """Manda una imagen al run de WandB.

    Con ``fast_dev_run=True``, Lightning cambia el logger del Trainer por un
    ``DummyLogger`` y no abre run. Si ``run_experiment`` abrió uno a mano,
    se usa ``wandb.run``.
    """
    logger = _logger_wandb(trainer)
    if logger is not None:
        logger.experiment.log({clave: imagen})
        return
    if wandb.run is not None:
        wandb.log({clave: imagen})
        return
    print(f"[sin WandB] no se sube {clave}: {titulo}")


def subir_imagen_wandb(
    trainer: pl.Trainer,
    clave: str,
    rejilla_chw: torch.Tensor,
    titulo: str,
) -> None:
    """Sube una rejilla CHW a WandB con la clave exacta del contrato."""
    arreglo = rejilla_chw.permute(1, 2, 0).numpy()
    _enviar_wandb(trainer, clave, wandb.Image(arreglo, caption=titulo), titulo)


def subir_figura_wandb(trainer: pl.Trainer, clave: str, figura: plt.Figure, titulo: str) -> None:
    """Sube una figura de matplotlib y la cierra."""
    try:
        _enviar_wandb(trainer, clave, wandb.Image(figura, caption=titulo), titulo)
    finally:
        plt.close(figura)


class ValReconstructionCallback(Callback):
    """16 imágenes de validación: original y reconstrucción, en ``val/reconstructions``."""

    def __init__(self, n_imagenes: int = 16) -> None:
        super().__init__()
        # El 16 es parte del contrato de la figura, no un hparam del cfg de entrenamiento.
        self.n_imagenes = int(n_imagenes)

    def on_validation_epoch_end(self, trainer: pl.Trainer, pl_module: pl.LightningModule) -> None:
        if trainer.sanity_checking or trainer.datamodule is None:
            return
        cargador = trainer.datamodule.val_dataloader()
        try:
            batch = next(iter(cargador))
        except StopIteration:
            print("val/reconstructions: el dataloader de val está vacío.")
            return
        x = batch[0]
        n = min(self.n_imagenes, int(x.size(0)))
        x = x[:n]
        estaba = pl_module.training
        pl_module.eval()
        try:
            with torch.no_grad():
                recon, _z = pl_module(x.to(pl_module.device))
        finally:
            pl_module.train(estaba)
        rejilla = rejilla_original_recon(x, recon)
        titulo = (
            f"val/reconstructions — época {trainer.current_epoch}. "
            "Arriba (o primeras filas) original; debajo reconstrucción."
        )
        subir_imagen_wandb(trainer, "val/reconstructions", rejilla, titulo)


class LatentTSNECallback(Callback):
    """t-SNE del latente de validación al acabar cada época. Clave ``val/tsne``.

    Decisión: se proyecta ``encode(x)``. En el VAE eso es ``mu``. El color es la
    clase de objeto (en val todas las imágenes son buenas).
    """

    def on_validation_epoch_end(self, trainer: pl.Trainer, pl_module: pl.LightningModule) -> None:
        if trainer.sanity_checking or trainer.datamodule is None:
            return
        cargador = trainer.datamodule.val_dataloader()
        latentes: list[torch.Tensor] = []
        clases: list[str] = []
        estaba = pl_module.training
        pl_module.eval()
        try:
            with torch.no_grad():
                for batch in cargador:
                    x = batch[0].to(pl_module.device)
                    z = pl_module.encode(x)
                    if z.ndim != 2:
                        raise ValueError(f"encode() debe devolver [B, z_dim], recibido {tuple(z.shape)}.")
                    latentes.append(z.detach().cpu())
                    clases.extend(str(clase) for clase in batch[2])
        finally:
            pl_module.train(estaba)
        if not latentes:
            print("val/tsne: validación vacía.")
            return
        matriz = torch.cat(latentes, dim=0).numpy()
        n = int(matriz.shape[0])
        if n < 4:
            # t-SNE no está definido con tan pocos puntos (típico de fast_dev_run con 1 imagen).
            print(f"val/tsne omitido: solo hay {n} latentes (hacen falta al menos 4).")
            return
        perplejidad = float(min(30, n - 1))
        inicio = "pca" if n >= 30 else "random"
        proyeccion = TSNE(
            n_components=2,
            perplexity=perplejidad,
            init=inicio,
            random_state=42,
        ).fit_transform(matriz)
        figura, eje = plt.subplots(figsize=(6, 5))
        for clase in sorted(set(clases)):
            mascara = [item == clase for item in clases]
            eje.scatter(
                proyeccion[mascara, 0],
                proyeccion[mascara, 1],
                s=18,
                alpha=0.85,
                label=clase,
                c=_COLORES_CLASE.get(clase, "gray"),
            )
        eje.set_title(f"val/tsne — época {trainer.current_epoch}")
        eje.set_xlabel("t-SNE 1")
        eje.set_ylabel("t-SNE 2")
        eje.legend(fontsize=8)
        figura.tight_layout()
        subir_figura_wandb(
            trainer,
            "val/tsne",
            figura,
            f"val/tsne — época {trainer.current_epoch}",
        )


class TestReconstructionCallback(Callback):
    """Hasta 8 buenas y 8 defectuosas del test. Clave ``test/reconstructions_good_defective``."""

    def __init__(self, n_por_grupo: int = 8) -> None:
        super().__init__()
        self.n_por_grupo = int(n_por_grupo)

    def on_test_epoch_end(self, trainer: pl.Trainer, pl_module: pl.LightningModule) -> None:
        if trainer.datamodule is None:
            return
        cargador = trainer.datamodule.test_dataloader()
        buenas: list[tuple[torch.Tensor, str, str]] = []
        malas: list[tuple[torch.Tensor, str, str]] = []
        for batch in cargador:
            imagenes, etiquetas, clases, subclases = batch
            for indice in range(int(imagenes.size(0))):
                ficha = (
                    imagenes[indice].detach().cpu(),
                    str(clases[indice]),
                    str(subclases[indice]),
                )
                if int(etiquetas[indice]) == 0 and len(buenas) < self.n_por_grupo:
                    buenas.append(ficha)
                elif int(etiquetas[indice]) == 1 and len(malas) < self.n_por_grupo:
                    malas.append(ficha)
            if len(buenas) >= self.n_por_grupo and len(malas) >= self.n_por_grupo:
                break
        seleccion = buenas + malas
        if not seleccion:
            print("test/reconstructions_good_defective: test vacío.")
            return
        x = torch.stack([ficha[0] for ficha in seleccion], dim=0)
        pl_module.eval()
        with torch.no_grad():
            recon, _z = pl_module(x.to(pl_module.device))
        rejilla = rejilla_original_recon(x, recon)
        leyenda = ", ".join(f"{clase}/{subclase}" for _img, clase, subclase in seleccion)
        if len(leyenda) > 280:
            leyenda = leyenda[:280] + "..."
        titulo = (
            f"test/reconstructions_good_defective — {len(buenas)} buenas + {len(malas)} defectuosas. "
            f"Orden: {leyenda}"
        )
        subir_imagen_wandb(trainer, "test/reconstructions_good_defective", rejilla, titulo)


publicar_en_tarea03(ValReconstructionCallback)
publicar_en_tarea03(LatentTSNECallback)
publicar_en_tarea03(TestReconstructionCallback)


## 7. Entrenamiento

**Dueño:** P1

**Qué hace:** `run_experiment(model_name, loss, fast_dev_run=False)` entrena un par arquitectura/pérdida y lo evalúa con el mejor checkpoint. `run_all(model_name)` recorre `l1`, `l2`, `ssim`, `ssim_l1` (sustituye a `--multirun`).

**Entradas:** nombre de modelo (`vae` o `u-net`) y nombre de pérdida. El resto sale de `get_cfg`.

**Salidas:** un dict con `val/loss` (mejor), `test/recon_error`, la ruta del checkpoint, `run_name` y `run_id`. Entre runs llama a `wandb.finish()` para no mezclar logs.

**No debe tocar otra persona:** el orden de las cuatro pérdidas, el `WandbLogger` (proyecto, group, tags, nombre) y el `ModelCheckpoint` que vigila `val/loss`. P2 llama a `run_all("vae")`; P3, a `run_all("u-net")`.

El argumento `extra_overrides` no forma parte del bucle de los 8 runs. Existe para que la celda de smoke (borrable) pueda cambiar el `_target_` a un autoencoder diminuto sin tocar `run_all`.

Los runs reales usan WandB online. Si el entorno trae `WANDB_MODE=offline` (lo pone el smoke), `log_model` se apaga: Lightning no permite subir checkpoints en offline.


In [ ]:
from __future__ import annotations

import os
from typing import Any

import hydra
import pytorch_lightning as pl
import wandb
from omegaconf import DictConfig
from pytorch_lightning import seed_everything
from pytorch_lightning.callbacks import Callback, ModelCheckpoint
from pytorch_lightning.loggers import WandbLogger


def _construir_trainer(
    cfg: DictConfig,
    fast_dev_run: bool,
) -> tuple[pl.Trainer, ModelCheckpoint, WandbLogger]:
    """Arma logger, checkpoint y Trainer a partir del cfg, sin hparams sueltos."""
    modo = os.environ.get("WANDB_MODE", "online")
    es_offline = modo in {"offline", "dryrun"}
    log_model_cfg = cfg.logger.log_model
    log_model: bool | str
    if isinstance(log_model_cfg, str):
        log_model = str(log_model_cfg)
    else:
        log_model = bool(log_model_cfg)
    if es_offline and log_model is not False:
        # WandbLogger(offline=True, log_model=True) lanza MisconfigurationException.
        # Solo ocurre si el proceso ya iba en offline (el smoke). Un run online no entra.
        log_model = False

    argumentos_logger: dict[str, Any] = {
        "project": str(cfg.logger.project),
        "name": str(cfg.logger.name),
        "group": str(cfg.logger.group),
        "tags": [str(etiqueta) for etiqueta in cfg.logger.tags],
        "log_model": log_model,
        "save_dir": "outputs",
        "offline": es_offline,
    }
    entidad = cfg.logger.entity if cfg.logger.entity is not None else os.environ.get("WANDB_ENTITY")
    if entidad:
        argumentos_logger["entity"] = str(entidad)
    registrador = WandbLogger(**argumentos_logger)

    checkpoint = ModelCheckpoint(
        monitor=str(cfg.trainer.checkpoint.monitor),
        mode=str(cfg.trainer.checkpoint.mode),
        save_top_k=int(cfg.trainer.checkpoint.save_top_k),
        filename="best-{epoch}",
        auto_insert_metric_name=False,
    )
    callbacks: list[Callback] = [
        checkpoint,
        ValReconstructionCallback(),
        LatentTSNECallback(),
        TestReconstructionCallback(),
    ]
    cada = 1 if fast_dev_run else int(cfg.trainer.log_every_n_steps)
    precision_cfg = cfg.trainer.precision
    precision: int | str = str(precision_cfg) if isinstance(precision_cfg, str) else int(precision_cfg)
    argumentos_trainer: dict[str, Any] = {
        "max_epochs": int(cfg.trainer.max_epochs),
        "accelerator": str(cfg.trainer.accelerator),
        "devices": int(cfg.trainer.devices),
        "precision": precision,
        "log_every_n_steps": cada,
        "callbacks": callbacks,
        "logger": registrador,
        "fast_dev_run": fast_dev_run,
        "default_root_dir": "outputs",
        "enable_progress_bar": True,
    }
    if not fast_dev_run:
        argumentos_trainer["num_sanity_val_steps"] = 2
    entrenador = pl.Trainer(**argumentos_trainer)
    return entrenador, checkpoint, registrador


def run_experiment(
    model_name: str,
    loss: str,
    fast_dev_run: bool = False,
    *,
    extra_overrides: list[str] | None = None,
) -> dict[str, Any]:
    """Entrena y evalúa un par (arquitectura, pérdida).

    Returns:
        Dict con ``modelo``, ``perdida``, ``run_name``, ``run_id``, ``clase``,
        ``val/loss`` (mejor score del checkpoint, o ``None``), ``test/recon_error``
        y ``checkpoint`` (ruta; vacía si ``fast_dev_run`` no llegó a guardar).

    ``wandb.finish()`` va en un ``finally`` para que un fallo no deje el run
    abierto y el siguiente se mezcle con él.
    """
    cfg = get_cfg(model_name, loss, extra_overrides=extra_overrides)
    seed_everything(int(cfg.seed), workers=True)
    datamodule = hydra.utils.instantiate(cfg.datamodule)
    modelo = hydra.utils.instantiate(cfg.model)
    if not isinstance(modelo, BaseAutoencoder):
        raise TypeError(f"El modelo instanciado no es BaseAutoencoder: {type(modelo)!r}.")

    modo_antes = os.environ.get("WANDB_MODE")
    es_offline = modo_antes in {"offline", "dryrun"}
    entrenador, checkpoint, _registrador = _construir_trainer(cfg, fast_dev_run)
    resumen: dict[str, Any] = {
        "modelo": model_name,
        "perdida": loss,
        "run_name": str(cfg.logger.name),
        "run_id": "",
        "clase": type(modelo).__name__,
        "val/loss": None,
        "test/recon_error": None,
        "checkpoint": "",
    }
    try:
        # fast_dev_run sustituye los loggers por DummyLogger y el WandbLogger no
        # llega a llamar a wandb.init. Abrimos el run a mano para que los
        # callbacks puedan subir figuras y para que finish() separe dos smoke.
        # Los 8 runs (fast_dev_run=False) no entran aquí.
        if fast_dev_run and wandb.run is None:
            kwargs_init: dict[str, Any] = {
                "project": str(cfg.logger.project),
                "name": str(cfg.logger.name),
                "group": str(cfg.logger.group),
                "tags": [str(etiqueta) for etiqueta in cfg.logger.tags],
                "job_type": "fast_dev_run",
            }
            entidad_run = cfg.logger.entity if cfg.logger.entity is not None else os.environ.get("WANDB_ENTITY")
            if entidad_run:
                kwargs_init["entity"] = str(entidad_run)
            wandb.init(**kwargs_init)
        entrenador.fit(modelo, datamodule=datamodule)
        ruta = checkpoint.best_model_path or ""
        if ruta:
            entrenador.test(modelo, datamodule=datamodule, ckpt_path=ruta)
        else:
            # fast_dev_run a menudo no escribe el .ckpt. Se evalúa el peso actual.
            entrenador.test(modelo, datamodule=datamodule)
        metricas = entrenador.callback_metrics
        mejor = checkpoint.best_model_score
        if mejor is not None:
            resumen["val/loss"] = float(mejor)
        elif "val/loss" in metricas:
            # fast_dev_run no guarda checkpoint, pero la métrica sí se calculó.
            resumen["val/loss"] = float(metricas["val/loss"])
        if "test/recon_error" in metricas:
            resumen["test/recon_error"] = float(metricas["test/recon_error"])
        resumen["checkpoint"] = ruta
        if wandb.run is not None:
            resumen["run_id"] = str(wandb.run.id)
        return resumen
    finally:
        wandb.finish()
        # WandbLogger(offline=True) escribe WANDB_MODE=dryrun al abrir el experimento.
        if es_offline:
            if modo_antes is None:
                os.environ.pop("WANDB_MODE", None)
            else:
                os.environ["WANDB_MODE"] = modo_antes


def run_all(model_name: str) -> list[dict[str, Any]]:
    """Lanza las cuatro pérdidas de una arquitectura. Sustituye a ``--multirun``."""
    if model_name not in {"vae", "u-net"}:
        raise ValueError(f"Modelo desconocido: {model_name!r}.")
    resultados: list[dict[str, Any]] = []
    for nombre_perdida in ("l1", "l2", "ssim", "ssim_l1"):
        print(f"=== {model_name} / {nombre_perdida} ===")
        resultados.append(run_experiment(model_name, nombre_perdida, fast_dev_run=False))
    return resultados


publicar_en_tarea03(run_experiment)
publicar_en_tarea03(run_all)


### Smoke test de infraestructura (BORRABLE / temporal)

Esta subsección **no** es uno de los 8 runs. Comprueba que Hydra instancia el modelo y el datamodule, que el `Trainer` hace `fit` y `test`, y que `wandb.finish()` deja runs distintos.

Si no hay MVTec en `DATA_DIR`, usa un datamodule falso de tensores `[0, 1]` 128×128×3, también borrable. El experimento se construye con `WandbLogger`, pero esta celda exporta `WANDB_MODE=offline` para no pedir API key. Lightning, en `fast_dev_run`, sustituye ese logger por uno mudo; `run_experiment` abre entonces el run a mano para que `wandb.finish()` separe los dos pasos. El camino de `run_all` no toca `WANDB_MODE` y sigue siendo WandB online.

Borrad esta subsección cuando P2 y P3 tengan un `forward` real. El smoke de cada arquitectura está reservado en las secciones 9 y 10.


In [ ]:
from __future__ import annotations

import os
from pathlib import Path
from typing import Any

import pytorch_lightning as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
import wandb
from torch.utils.data import DataLoader, Dataset


class TinyAE(BaseAutoencoder):
    """Autoencoder mínimo de verdad, solo para el smoke. Borradlo con esta celda.

    Encoder: conv + pool global + lineal a ``z_dim``. Decoder: lineal, dos
    conv-transpose y, al final, sigmoid (después de igualar el H×W de la entrada).
    """

    def __init__(
        self,
        z_dim: int,
        loss: str,
        lr: float,
        alpha: float,
        name: str = "tiny",
        **kwargs: Any,
    ) -> None:
        super().__init__(z_dim=z_dim, loss=loss, lr=lr, alpha=alpha, name=name, **kwargs)
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 8, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.fc_z = nn.Linear(8, int(z_dim))
        self.fc_dec = nn.Linear(int(z_dim), 8 * 4 * 4)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(8, 8, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(8, 3, kernel_size=4, stride=2, padding=1),
        )

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc_z(self.encoder(x).flatten(1))

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        z = self.encode(x)
        h = self.fc_dec(z).view(-1, 8, 4, 4)
        h = self.decoder(h)
        h = F.interpolate(h, size=x.shape[-2:], mode="bilinear", align_corners=False)
        return torch.sigmoid(h), z


class _DatasetSintetico(Dataset):
    """Unas pocas imágenes sintéticas con clase y subclase. Solo para el smoke."""

    def __init__(
        self,
        n: int,
        img_size: int,
        seed: int,
        classes: list[str],
        solo_buenas: bool,
    ) -> None:
        super().__init__()
        generador = torch.Generator().manual_seed(int(seed))
        self.imagenes = torch.rand(int(n), 3, int(img_size), int(img_size), generator=generador)
        self.clases = [classes[i % len(classes)] for i in range(int(n))]
        if solo_buenas:
            self.etiquetas = [0] * int(n)
            self.subclases = ["good"] * int(n)
        else:
            self.etiquetas = [0 if i % 2 == 0 else 1 for i in range(int(n))]
            self.subclases = [
                "good" if etiqueta == 0 else "synthetic_defect" for etiqueta in self.etiquetas
            ]

    def __len__(self) -> int:
        return int(self.imagenes.shape[0])

    def __getitem__(self, index: int) -> tuple[torch.Tensor, int, str, str]:
        return (
            self.imagenes[index],
            self.etiquetas[index],
            self.clases[index],
            self.subclases[index],
        )


class SmokeDataModule(pl.LightningDataModule):
    """Datamodule falso. Acepta la firma de ``MVTecDataModule`` porque Hydra se la pasa.

    BORRABLE. No lo uséis en los 8 runs.
    """

    def __init__(
        self,
        data_dir: str = "",
        img_size: int = 128,
        batch_size: int = 4,
        num_workers: int = 0,
        val_split: float = 0.1,
        classes: list[str] | None = None,
        seed: int = 42,
        **kwargs: Any,
    ) -> None:
        super().__init__()
        self.img_size = int(img_size)
        # Un batch pequeño: fast_dev_run solo da un paso y tiene que caber en CPU.
        self.batch_size = max(1, min(int(batch_size), 8))
        self.num_workers = 0
        self.seed = int(seed)
        self.classes = list(classes or ["cable", "capsule", "screw", "transistor"])
        self.train_set: Dataset | None = None
        self.val_set: Dataset | None = None
        self.test_set: Dataset | None = None

    def setup(self, stage: str | None = None) -> None:
        if stage in (None, "fit", "validate"):
            self.train_set = _DatasetSintetico(8, self.img_size, self.seed, self.classes, True)
            self.val_set = _DatasetSintetico(8, self.img_size, self.seed + 1, self.classes, True)
        if stage in (None, "test"):
            self.test_set = _DatasetSintetico(8, self.img_size, self.seed + 2, self.classes, False)

    def _loader(self, dataset: Dataset | None, shuffle: bool) -> DataLoader[Any]:
        if dataset is None:
            raise RuntimeError("setup() no fue llamado.")
        return DataLoader(
            dataset,
            batch_size=self.batch_size,
            shuffle=shuffle,
            num_workers=0,
            collate_fn=collate_mvtec,
        )

    def train_dataloader(self) -> DataLoader[Any]:
        return self._loader(self.train_set, shuffle=True)

    def val_dataloader(self) -> DataLoader[Any]:
        return self._loader(self.val_set, shuffle=False)

    def test_dataloader(self) -> DataLoader[Any]:
        return self._loader(self.test_set, shuffle=False)


publicar_en_tarea03(TinyAE)
publicar_en_tarea03(SmokeDataModule)

if not RUN_SMOKE:
    print("RUN_SMOKE=False: smoke test omitido.")
else:
    hay_mvtec = (Path(DATA_DIR) / "cable" / "train" / "good").is_dir()
    overrides = [
        "model._target_=tarea03.TinyAE",
        "logger.log_model=false",
    ]
    if not hay_mvtec:
        overrides.append("datamodule._target_=tarea03.SmokeDataModule")
        print("MVTec no está en DATA_DIR: smoke con SmokeDataModule (BORRABLE).")
    else:
        print("MVTec detectado: smoke con el DataModule real y fast_dev_run.")

    # Offline explícito: no exige API key y aun así pasa por WandbLogger + wandb.finish().
    # Los 8 runs reales no ejecutan este bloque, así que siguen en WandB online.
    # Alternativa comentada si offline no arrancara en algún kernel:
    #   from pytorch_lightning.loggers import CSVLogger
    #   (habría que bifurcar run_experiment; no es el camino de los 8 runs)
    modo_previo = os.environ.get("WANDB_MODE")
    os.environ["WANDB_MODE"] = "offline"
    try:
        resumen_l1 = run_experiment("vae", "l1", fast_dev_run=True, extra_overrides=overrides)
        resumen_l2 = run_experiment("vae", "l2", fast_dev_run=True, extra_overrides=overrides)
        assert resumen_l1["clase"] == "TinyAE", resumen_l1
        assert resumen_l2["clase"] == "TinyAE", resumen_l2
        assert resumen_l1["run_name"] == "vae_l1_z128", resumen_l1
        assert resumen_l2["run_name"] == "vae_l2_z128", resumen_l2
        assert resumen_l1["run_name"] != resumen_l2["run_name"]
        assert resumen_l1["run_id"] and resumen_l1["run_id"] != resumen_l2["run_id"], (
            resumen_l1,
            resumen_l2,
        )
        assert resumen_l1["test/recon_error"] is not None
        assert resumen_l2["test/recon_error"] is not None
        assert wandb.run is None, "wandb.finish() no cerró el run"
        print("SMOKE OK", resumen_l1["run_name"], resumen_l1["run_id"], resumen_l2["run_name"], resumen_l2["run_id"])
    finally:
        if modo_previo is None:
            os.environ.pop("WANDB_MODE", None)
        else:
            os.environ["WANDB_MODE"] = modo_previo


## 8. Análisis

**Dueño:** P1

**Qué hace:** `compute_errors` mide el MSE por imagen de test; `plot_error_histograms` dibuja normal vs defectuoso por subclase y lo sube a WandB como `test/error_histograms`. `comparar_runs` baja del proyecto los runs (curvas e imágenes) para las secciones 9, 10 y 11.

**Entradas:** un modelo ya entrenado y el datamodule, o la API de WandB si `RUN_COMPARISON` está activa.

**Salidas:** un `DataFrame` con columnas `error`, `label`, `clase`, `subclase`. Figuras que viven en WandB.

**No debe tocar otra persona:** los nombres de columna y la clave `test/error_histograms`. P2 y P3 llaman a estas funciones; no rehacen el histograma en local para pegarlo en la memoria.

Decisión: la curva train/val de la comparación se dibuja a partir del **historial descargado** con `wandb.Api` (el enunciado pide graficarla aquí). Las reconstrucciones, el t-SNE y los histogramas no se recalculan: se descarga el fichero que el run ya subió. De cada clave se enseña la última época (el fichero con el paso más alto), no las 100.


In [ ]:
from __future__ import annotations

import math
import os
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import pandas as pd
import pytorch_lightning as pl
import torch
import wandb
from omegaconf import OmegaConf
from tqdm.auto import tqdm


@torch.no_grad()
def compute_errors(
    model: pl.LightningModule,
    datamodule: pl.LightningDataModule,
) -> pd.DataFrame:
    """MSE de reconstrucción por imagen en el test oficial.

    Columnas: ``error``, ``label`` (0/1), ``clase``, ``subclase``.
    """
    model.eval()
    datamodule.setup("test")
    try:
        dispositivo = next(model.parameters()).device
    except StopIteration:
        dispositivo = torch.device("cpu")
    filas: list[dict[str, Any]] = []
    cargador = datamodule.test_dataloader()
    for batch in tqdm(cargador, desc="compute_errors"):
        x, etiqueta, clase, subclase = batch
        recon, _z = model(x.to(dispositivo))
        error = mse_por_imagen(recon, x).detach().cpu()
        for indice in range(int(x.size(0))):
            filas.append(
                {
                    "error": float(error[indice]),
                    "label": int(etiqueta[indice]),
                    "clase": str(clase[indice]),
                    "subclase": str(subclase[indice]),
                }
            )
    tabla = pd.DataFrame(filas, columns=["error", "label", "clase", "subclase"])
    return tabla


def _proyecto_wandb() -> str:
    """Nombre de proyecto tal como está en ``conf/logger/wandb.yaml``."""
    ruta = Path("conf/logger/wandb.yaml")
    if not ruta.is_file():
        raise FileNotFoundError("Falta conf/logger/wandb.yaml. Ejecuta las celdas %%writefile.")
    return str(OmegaConf.load(ruta).project)


def plot_error_histograms(df: pd.DataFrame, *, run_name: str = "histogramas") -> None:
    """Histograma de error de las buenas contra cada subclase de defecto.

    La figura se sube a WandB con la clave ``test/error_histograms``.
    Si no hay un run abierto, esta función abre uno en el proyecto de la tarea
    y lo cierra al terminar.
    """
    columnas = {"error", "label", "clase", "subclase"}
    if not columnas.issubset(df.columns):
        raise ValueError(f"El DataFrame debe tener {sorted(columnas)}; tiene {list(df.columns)}.")
    if df.empty:
        raise ValueError("El DataFrame de errores está vacío.")

    buenas = df.loc[df["label"] == 0, "error"].to_numpy()
    subclases = sorted(str(valor) for valor in df["subclase"].unique() if str(valor) != "good")
    if not subclases:
        subclases = ["(sin defectos)"]
    n_paneles = len(subclases)
    columnas_fig = 4
    filas_fig = int(math.ceil(n_paneles / columnas_fig))
    figura, ejes = plt.subplots(
        filas_fig,
        columnas_fig,
        figsize=(4 * columnas_fig, 3 * filas_fig),
        squeeze=False,
    )
    minimo = float(df["error"].min())
    maximo = float(df["error"].max())
    if math.isclose(minimo, maximo):
        maximo = minimo + 1e-6
    for indice, subclase in enumerate(subclases):
        eje = ejes[indice // columnas_fig][indice % columnas_fig]
        defectos = df.loc[df["subclase"] == subclase, "error"].to_numpy()
        if len(buenas):
            eje.hist(buenas, bins=20, range=(minimo, maximo), alpha=0.65, label="good", color="steelblue")
        if len(defectos):
            eje.hist(defectos, bins=20, range=(minimo, maximo), alpha=0.65, label=subclase, color="crimson")
        eje.set_title(subclase, fontsize=9)
        eje.set_xlabel("MSE")
        eje.legend(fontsize=7)
    for indice in range(n_paneles, filas_fig * columnas_fig):
        ejes[indice // columnas_fig][indice % columnas_fig].axis("off")
    figura.suptitle("Error de reconstrucción: normal vs defectuoso")
    figura.tight_layout()

    abierto_aqui = wandb.run is None
    if abierto_aqui:
        wandb.init(
            project=_proyecto_wandb(),
            entity=os.environ.get("WANDB_ENTITY") or None,
            name=run_name,
            group="analisis",
            job_type="histogramas",
            tags=["histogramas"],
        )
    try:
        wandb.log({"test/error_histograms": wandb.Image(figura)})
        print(f"Figura subida a WandB como test/error_histograms (run {run_name}).")
    finally:
        plt.close(figura)
        if abierto_aqui:
            wandb.finish()


publicar_en_tarea03(compute_errors)
publicar_en_tarea03(plot_error_histograms)


In [ ]:
from __future__ import annotations

import re
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import os
import pandas as pd
import wandb
from omegaconf import OmegaConf

_CLAVES_FIGURA = (
    "test/reconstructions_good_defective",
    "test/error_histograms",
    "val/reconstructions",
    "val/tsne",
)
_PERDIDAS_RUN = ("l1", "l2", "ssim", "ssim_l1")


def _ruta_proyecto_wandb() -> str:
    """``entidad/proyecto``. La entidad sale de WANDB_ENTITY, del YAML o de la cuenta."""
    cfg_logger = OmegaConf.load(Path("conf/logger/wandb.yaml"))
    proyecto = str(cfg_logger.project)
    entidad = os.environ.get("WANDB_ENTITY")
    if not entidad and cfg_logger.entity not in (None, "null"):
        entidad = str(cfg_logger.entity)
    if not entidad:
        entidad = wandb.Api().default_entity
    if not entidad:
        raise RuntimeError(
            "No hay entidad de WandB. Exporta WANDB_ENTITY o rellena entity en conf/logger/wandb.yaml."
        )
    return f"{entidad}/{proyecto}"


def _nombre_de_run(nombre: str, modelos: tuple[str, ...], z_dim: int) -> bool:
    for modelo in modelos:
        prefijo = f"{modelo}_"
        if nombre.startswith(prefijo) and nombre.endswith(f"_z{z_dim}"):
            resto = nombre[len(prefijo) : -len(f"_z{z_dim}")]
            if resto in _PERDIDAS_RUN:
                return True
    return False


def _serie(historial: pd.DataFrame, clave: str) -> pd.Series:
    for candidata in (clave, f"{clave}_epoch"):
        if candidata in historial.columns:
            return historial[candidata].dropna().reset_index(drop=True)
    return pd.Series(dtype=float)


def _categoria_fichero(nombre: str) -> str | None:
    texto = nombre.lower()
    if not texto.endswith((".png", ".jpg", ".jpeg", ".webp")):
        return None
    for clave in _CLAVES_FIGURA:
        compacta = clave.lower().replace("/", "")
        if clave in texto or clave.replace("/", "_") in texto or compacta in texto.replace("/", "").replace("_", ""):
            return clave
    return None


def _paso_fichero(nombre: str) -> int:
    numeros = re.findall(r"(\d+)", Path(nombre).stem)
    return int(numeros[-1]) if numeros else -1


def comparar_runs(modelos: tuple[str, ...] = ("vae", "u-net")) -> pd.DataFrame:
    """Baja runs del proyecto y muestra curvas e imágenes que ya están en WandB.

    Las curvas salen del historial de métricas (``train/loss``, ``val/loss``).
    Reconstrucciones, t-SNE e histogramas se descargan; no se vuelve a hacer
    un forward. De cada clave se muestra el fichero de paso más alto.
    """
    if not modelos:
        raise ValueError("Indica al menos un modelo: 'vae', 'u-net' o ambos.")
    z_dim = int(get_cfg(modelos[0], "l1").model.z_dim)
    ruta = _ruta_proyecto_wandb()
    api = wandb.Api()
    try:
        candidatos = list(api.runs(ruta, filters={"group": {"$in": list(modelos)}}))
    except Exception as error:
        print(f"Filtro de la API no disponible ({error}). Se listan los runs del proyecto.")
        candidatos = list(api.runs(ruta))

    elegidos: dict[str, Any] = {}
    for run in candidatos:
        nombre = str(run.name or "")
        if not _nombre_de_run(nombre, modelos, z_dim):
            continue
        anterior = elegidos.get(nombre)
        if anterior is None or str(run.created_at) > str(anterior.created_at):
            elegidos[nombre] = run

    esperados = [f"{modelo}_{perdida}_z{z_dim}" for modelo in modelos for perdida in _PERDIDAS_RUN]
    faltan = [nombre for nombre in esperados if nombre not in elegidos]
    print(f"Proyecto {ruta}. Runs usados: {sorted(elegidos)}.")
    if faltan:
        print("Aún no están:", ", ".join(faltan))

    from IPython.display import display

    filas: list[dict[str, Any]] = []
    for modelo in modelos:
        figura, ejes = plt.subplots(1, 2, figsize=(12, 4), sharex=False)
        for perdida in _PERDIDAS_RUN:
            nombre = f"{modelo}_{perdida}_z{z_dim}"
            run = elegidos.get(nombre)
            if run is None:
                continue
            historial = run.history(samples=10_000, pandas=True)
            for eje, clave in zip(ejes, ("train/loss", "val/loss")):
                serie = _serie(historial, clave)
                if len(serie):
                    eje.plot(serie.index, serie.to_numpy(), label=perdida)
            resumen = run.summary
            filas.append(
                {
                    "run": nombre,
                    "group": modelo,
                    "train/loss": resumen.get("train/loss", resumen.get("train/loss_epoch")),
                    "val/loss": resumen.get("val/loss", resumen.get("val/loss_epoch")),
                    "test/recon_error": resumen.get("test/recon_error"),
                    "train/kl": resumen.get("train/kl"),
                    "val/kl": resumen.get("val/kl"),
                }
            )
            _mostrar_imagenes_run(run)
        for eje, titulo in zip(ejes, ("train/loss", "val/loss")):
            eje.set_title(f"{modelo} — {titulo}")
            eje.set_xlabel("época (puntos logueados)")
            eje.set_ylabel(titulo)
            eje.legend(fontsize=8)
        figura.tight_layout()
        display(figura)
        plt.close(figura)

    tabla = pd.DataFrame(filas)
    if not tabla.empty:
        display(tabla)
    return tabla


def _mostrar_imagenes_run(run: Any) -> None:
    """Descarga la última imagen de cada clave y la enseña con IPython."""
    por_clave: dict[str, Any] = {}
    for fichero in run.files():
        categoria = _categoria_fichero(str(fichero.name))
        if categoria is None:
            continue
        previo = por_clave.get(categoria)
        if previo is None or _paso_fichero(str(fichero.name)) >= _paso_fichero(str(previo.name)):
            por_clave[categoria] = fichero
    if not por_clave:
        print(f"{run.name}: no hay imágenes de reconstrucción, t-SNE o histogramas.")
        return
    from IPython.display import Image as ImagenIP
    from IPython.display import display

    destino_raiz = Path("outputs/wandb_media") / str(run.id)
    destino_raiz.mkdir(parents=True, exist_ok=True)
    for clave in _CLAVES_FIGURA:
        fichero = por_clave.get(clave)
        if fichero is None:
            continue
        descargado = fichero.download(root=str(destino_raiz), replace=True)
        ruta = Path(getattr(descargado, "name", descargado))
        if hasattr(descargado, "close"):
            descargado.close()
        print(f"{run.name} — {clave} — {ruta.name}")
        display(ImagenIP(filename=str(ruta)))


publicar_en_tarea03(comparar_runs)


## 9. VAE

**Dueño:** P2

**Qué hace:** esta sección es un stub. P2 tiene que dejar un `VAE` entrenable que respete el contrato de `BaseAutoencoder`.

**Entradas:** el cfg `conf/model/vae.yaml` (`z_dim`, `loss`, `lr`, `alpha`, `beta`, `name`). El datamodule y `run_all("vae")` ya están.

**Salidas:** cuatro runs `vae_{l1,l2,ssim,ssim_l1}_z128`, con `train/kl` y `val/kl`, más el análisis crítico.

**No debe tocar otra persona:** P1 no reescribe el encoder. P3 no mete skips aquí. No cambiéis los nombres de métrica ni `build_loss`.

### Qué hay que implementar

- Encoder convolucional hasta `mu` y `logvar`, los dos con forma `[B, z_dim]`.
- Guardar `self.mu` y `self.logvar` en cada `forward`.
- `encode(x)` devuelve **`mu`** (lo usa el t-SNE).
- Reparametrización: `z = mu + eps * exp(0.5 * logvar)`, con `eps ~ N(0, I)`.
- `forward` devuelve `(recon, z)` donde `z` es esa muestra. `recon` termina en **sigmoid** y copia el `H×W` de `x`.
- `extra_loss` devuelve `beta * KL` y asigna `self.last_kl` a la KL **sin** beta:

```text
KL = -0.5 * sum(1 + logvar - mu^2 - exp(logvar), dim=1).mean()
```

- Borrad `self._parametro_stub` cuando existan capas de verdad. Solo está para que Adam no reciba una lista vacía.
- `beta`, `z_dim`, `lr`, `loss` y `alpha` salen de `self.hparams` / `self.beta`. No los hardcodeéis.

La clase se importa y se construye con Hydra. `encode`, `forward` y `extra_loss` lanzan `NotImplementedError` hasta que P2 los rellene.


In [ ]:
from __future__ import annotations

from typing import Any

import hydra
import torch
import torch.nn as nn


class VAE(BaseAutoencoder):
    """VAE (P2). Stub: se construye, pero ``encode`` / ``forward`` / ``extra_loss`` lanzan.

    Ver el markdown de esta sección para el contrato (mu, logvar, reparametrización,
    sigmoid y KL con ``beta``).
    """

    def __init__(
        self,
        z_dim: int,
        loss: str,
        lr: float,
        alpha: float,
        beta: float,
        name: str = "vae",
        **kwargs: Any,
    ) -> None:
        super().__init__(
            z_dim=z_dim,
            loss=loss,
            lr=lr,
            alpha=alpha,
            name=name,
            beta=beta,
            **kwargs,
        )
        self.beta = float(beta)
        self.mu: torch.Tensor | None = None
        self.logvar: torch.Tensor | None = None
        # TODO(P2): borrar este parámetro cuando el encoder y el decoder existan.
        self._parametro_stub = nn.Parameter(torch.zeros(1))

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        """TODO(P2): devolver ``mu`` con forma ``[B, z_dim]``."""
        raise NotImplementedError("P2: implementar VAE.encode (debe devolver mu).")

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """TODO(P2): reparametrizar, decodificar con sigmoid y devolver ``(recon, z)``."""
        raise NotImplementedError("P2: implementar VAE.forward -> (recon, z).")

    def extra_loss(
        self,
        recon: torch.Tensor,
        x: torch.Tensor,
        z: torch.Tensor,
    ) -> torch.Tensor:
        """TODO(P2): devolver ``beta * KL`` y guardar la KL sin beta en ``self.last_kl``.

        ``KL = -0.5 * sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1).mean()``.
        """
        raise NotImplementedError("P2: implementar la KL del VAE en extra_loss.")


publicar_en_tarea03(VAE)

cfg_vae_real = get_cfg("vae", "ssim")
modelo_vae = hydra.utils.instantiate(cfg_vae_real.model)
assert hydra.utils.get_class("tarea03.VAE") is VAE
assert isinstance(modelo_vae, VAE)
assert modelo_vae.hparams.loss == "ssim"
assert float(modelo_vae.beta) == 1.0
assert int(modelo_vae.hparams.z_dim) == 128
assert modelo_vae.mu is None and modelo_vae.logvar is None
try:
    modelo_vae.encode(torch.zeros(1, 3, 128, 128))
except NotImplementedError:
    print("VAE resoluble por Hydra. encode/forward siguen en stub (P2).")
else:
    raise AssertionError("El stub de VAE.encode debía lanzar NotImplementedError")


In [ ]:
from __future__ import annotations

import torch

# Celda reservada a P2. Con el stub no entrena: solo avisa.
# Cuando encode/forward existan, sustituid el bloque interno por:
#   run_experiment("vae", "l1", fast_dev_run=True)
if not RUN_SMOKE:
    print("RUN_SMOKE=False: smoke del VAE omitido.")
else:
    modelo_smoke_vae = hydra.utils.instantiate(get_cfg("vae", "l1").model)
    lote_vae = torch.rand(2, 3, 128, 128)
    try:
        recon_vae, z_vae = modelo_smoke_vae(lote_vae)
    except NotImplementedError as error:
        print(f"TODO(P2): smoke de arquitectura omitido ({error}).")
    else:
        assert recon_vae.shape == lote_vae.shape, tuple(recon_vae.shape)
        assert z_vae.shape == (lote_vae.shape[0], int(modelo_smoke_vae.hparams.z_dim))
        assert float(recon_vae.min()) >= 0.0 and float(recon_vae.max()) <= 1.0
        print("Smoke de arquitectura VAE: ok", tuple(recon_vae.shape), tuple(z_vae.shape))


In [ ]:
from __future__ import annotations

if RUN_TRAINING:
    resultados_vae = run_all("vae")
    print(resultados_vae)
else:
    resultados_vae = []
    print("RUN_TRAINING=False: no se lanza run_all('vae').")


In [ ]:
from __future__ import annotations

import pandas as pd

# Comparación de las 4 pérdidas del VAE. Las curvas e imágenes salen de WandB
# (comparar_runs). La tabla local solo aparece si acabáis de entrenar en este kernel.
if RUN_COMPARISON:
    tabla_vae = comparar_runs(("vae",))
elif RUN_TRAINING and resultados_vae:
    from IPython.display import display

    display(pd.DataFrame(resultados_vae))
else:
    print(
        "Comparación del VAE omitida. Tras los 4 runs, activa RUN_COMPARISON "
        "para bajar curvas e imágenes de WandB."
    )


In [ ]:
from __future__ import annotations

import hydra

if RUN_TRAINING and resultados_vae:
    for resumen_vae in resultados_vae:
        ruta_ckpt = str(resumen_vae.get("checkpoint") or "")
        if not ruta_ckpt:
            print(f"Sin checkpoint para {resumen_vae.get('run_name')}; no hay histograma.")
            continue
        modelo_hist = VAE.load_from_checkpoint(ruta_ckpt, map_location=DEVICE)
        cfg_hist = get_cfg("vae", str(resumen_vae["perdida"]))
        dm_hist = hydra.utils.instantiate(cfg_hist.datamodule)
        df_vae = compute_errors(modelo_hist.to(DEVICE), dm_hist)
        plot_error_histograms(df_vae, run_name=f"hist_{resumen_vae['run_name']}")
else:
    print("Histogramas del VAE omitidos (hace falta RUN_TRAINING y un checkpoint).")


### Análisis crítico del VAE

**Dueño:** P2. Responded con las figuras que `comparar_runs` baja de WandB, no con un forward nuevo.

1. Con `beta=1`, ¿la KL se queda en un valor alto o el posterior se aplasta (posterior collapse)? Mirad `train/kl` y `val/kl` en las cuatro pérdidas.
2. ¿Qué pérdida separa mejor el histograma de las imágenes buenas y el de cada subclase de defecto? ¿Coincide con la que gana en `val/loss`? Recordad que la validación **solo tiene buenas**.
3. En las reconstrucciones de val, ¿L2 emborrona, L1 aplana el color y SSIM aguanta mejor los bordes? ¿`ssim_l1` se queda en medio?
4. En test, ¿alguna pérdida reconstruye el defecto tan bien que `test/recon_error` de las defectuosas no se separa del de las buenas?
5. ¿El mismo `z_dim=128` sobra o se queda corto para las cuatro clases juntas (cable frente a screw)?

**Respuestas (P2):**

- (pendiente)


## 10. U-Net autoencoder

**Dueño:** P3

**Qué hace:** stub de `UNetAE`. P3 implementa el encoder-decoder con skip connections y un cuello del que sale un vector `z` de tamaño `z_dim`.

**Entradas:** `conf/model/u-net.yaml` (`z_dim`, `loss`, `lr`, `alpha`, `base_channels`, `depth`, `name`).

**Salidas:** cuatro runs `u-net_{l1,l2,ssim,ssim_l1}_z128` y el análisis crítico. No hay `train/kl`: no sobreescribáis `extra_loss`.

**No debe tocar otra persona:** P2 no añade KL aquí. P1 no redibuja los callbacks. No cambiéis el `_target_` ni el nombre `u-net` (con guion): el grupo Hydra y el `group` de WandB dependen de él.

### Qué hay que implementar

- Encoder-decoder simétrico. Los skips son por **concatenación de canales** (U-Net clásica). Si usáis suma, anotadlo en el análisis.
- `base_channels` es el ancho del primer bloque; cada nivel siguiente lo dobla. `depth` es el número de bajadas de resolución (otras tantas subidas). Con imagen 128 y `depth=4`, el mapa anterior al cuello queda en 8×8 si cada bloque divide por 2. Podéis precisar esta lectura en un comentario si el diseño se desvía, pero el número tiene que salir del cfg.
- El cuello se proyecta a un vector `z` de forma `[B, z_dim]` (pool + lineal, o el aplanado si cuadra). `encode` devuelve **ese vector**, no un mapa espacial. `forward` devuelve `(recon, z)`.
- El decoder parte de `z`, vuelve al mapa espacial, aplica los skips y **termina en sigmoid**. `recon` debe tener el mismo `H×W` que `x` (no hardcodeéis 128 si podéis leerlo de `x`).
- Borrad `self._parametro_stub` al crear las capas.

### Riesgo de los skips

Un skip copia detalle espacial de la entrada a la salida. La U-Net puede reconstruir también el **defecto**, y entonces el error de las defectuosas se parece al de las buenas: el cuello de `z_dim` no está forzando una representación compacta. Comprobadlo en `test/reconstructions_good_defective` y en los histogramas antes de dar por buena la arquitectura.


In [ ]:
from __future__ import annotations

from typing import Any

import hydra
import torch
import torch.nn as nn


class UNetAE(BaseAutoencoder):
    """U-Net autoencoder con skips (P3). Stub: ``encode`` y ``forward`` lanzan.

    El cuello tiene que producir ``z`` de tamaño ``z_dim`` (cfg). No hay KL.
    El riesgo de reconstruir también la anomalía está explicado en el markdown.
    """

    def __init__(
        self,
        z_dim: int,
        loss: str,
        lr: float,
        alpha: float,
        base_channels: int,
        depth: int,
        name: str = "u-net",
        **kwargs: Any,
    ) -> None:
        super().__init__(
            z_dim=z_dim,
            loss=loss,
            lr=lr,
            alpha=alpha,
            name=name,
            base_channels=base_channels,
            depth=depth,
            **kwargs,
        )
        self.base_channels = int(base_channels)
        self.depth = int(depth)
        # TODO(P3): borrar este parámetro cuando existan los bloques de la U-Net.
        self._parametro_stub = nn.Parameter(torch.zeros(1))

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        """TODO(P3): vector del cuello de botella, forma ``[B, z_dim]``."""
        raise NotImplementedError("P3: implementar UNetAE.encode -> z [B, z_dim].")

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """TODO(P3): decoder con skips y sigmoid. Devolver ``(recon, z)``."""
        raise NotImplementedError("P3: implementar UNetAE.forward -> (recon, z).")


publicar_en_tarea03(UNetAE)

cfg_unet_real = get_cfg("u-net", "l1")
modelo_unet = hydra.utils.instantiate(cfg_unet_real.model)
assert hydra.utils.get_class("tarea03.UNetAE") is UNetAE
assert isinstance(modelo_unet, UNetAE)
assert modelo_unet.hparams.loss == "l1"
assert modelo_unet.base_channels == 32
assert modelo_unet.depth == 4
assert int(modelo_unet.hparams.z_dim) == 128
try:
    modelo_unet.encode(torch.zeros(1, 3, 128, 128))
except NotImplementedError:
    print("UNetAE resoluble por Hydra. encode/forward siguen en stub (P3).")
else:
    raise AssertionError("El stub de UNetAE.encode debía lanzar NotImplementedError")


In [ ]:
from __future__ import annotations

import torch

# Celda reservada a P3. Misma idea que el smoke del VAE.
# Cuando forward exista:
#   run_experiment("u-net", "l1", fast_dev_run=True)
if not RUN_SMOKE:
    print("RUN_SMOKE=False: smoke de la U-Net omitido.")
else:
    modelo_smoke_unet = hydra.utils.instantiate(get_cfg("u-net", "l1").model)
    lote_unet = torch.rand(2, 3, 128, 128)
    try:
        recon_unet, z_unet = modelo_smoke_unet(lote_unet)
    except NotImplementedError as error:
        print(f"TODO(P3): smoke de arquitectura omitido ({error}).")
    else:
        assert recon_unet.shape == lote_unet.shape, tuple(recon_unet.shape)
        assert z_unet.shape == (lote_unet.shape[0], int(modelo_smoke_unet.hparams.z_dim))
        assert float(recon_unet.min()) >= 0.0 and float(recon_unet.max()) <= 1.0
        print("Smoke de arquitectura U-Net: ok", tuple(recon_unet.shape), tuple(z_unet.shape))


In [ ]:
from __future__ import annotations

if RUN_TRAINING:
    resultados_unet = run_all("u-net")
    print(resultados_unet)
else:
    resultados_unet = []
    print("RUN_TRAINING=False: no se lanza run_all('u-net').")


In [ ]:
from __future__ import annotations

import pandas as pd

if RUN_COMPARISON:
    tabla_unet = comparar_runs(("u-net",))
elif RUN_TRAINING and resultados_unet:
    from IPython.display import display

    display(pd.DataFrame(resultados_unet))
else:
    print(
        "Comparación de la U-Net omitida. Tras los 4 runs, activa RUN_COMPARISON "
        "para bajar curvas e imágenes de WandB."
    )


In [ ]:
from __future__ import annotations

import hydra

if RUN_TRAINING and resultados_unet:
    for resumen_unet in resultados_unet:
        ruta_ckpt = str(resumen_unet.get("checkpoint") or "")
        if not ruta_ckpt:
            print(f"Sin checkpoint para {resumen_unet.get('run_name')}; no hay histograma.")
            continue
        modelo_hist = UNetAE.load_from_checkpoint(ruta_ckpt, map_location=DEVICE)
        cfg_hist = get_cfg("u-net", str(resumen_unet["perdida"]))
        dm_hist = hydra.utils.instantiate(cfg_hist.datamodule)
        df_unet = compute_errors(modelo_hist.to(DEVICE), dm_hist)
        plot_error_histograms(df_unet, run_name=f"hist_{resumen_unet['run_name']}")
else:
    print("Histogramas de la U-Net omitidos (hace falta RUN_TRAINING y un checkpoint).")


### Análisis crítico de la U-Net

**Dueño:** P3. Usad las imágenes descargadas de WandB.

1. ¿Los skips reconstruyen el defecto? Comparad, para la misma pérdida, el hueco good/defectuoso de la U-Net con el del VAE.
2. ¿Hay subclases (rayas finas en screw, patas en transistor, cortes en cable) donde la U-Net copia el fallo y el error no sube?
3. Si subís `depth` o `base_channels`, ¿empeora la detección porque llega más detalle por los skips? No hace falta barrer el cfg en esta entrega: argumentad con las reconstrucciones del run que sí entrenasteis.
4. ¿El vector `z` aporta algo, o la imagen viaja casi toda por los skips? Una pista: si tapar el cuello (poner `z` a cero en un experimento aparte) casi no cambia `recon`, el cuello está de adorno.
5. ¿SSIM, que premia estructura, acentúa el problema de copiar texturas periódicas frente a L1 o L2?

**Respuestas (P3):**

- (pendiente)


## 11. Comparación

**Dueño:** todos

**Qué hace:** con `RUN_COMPARISON = True`, `comparar_runs(("vae", "u-net"))` trae los 8 runs del proyecto `tarea03-autoencoder-mvtec`, dibuja las curvas train/val de las 4 pérdidas por modelo y descarga las imágenes subidas a WandB (reconstrucciones, t-SNE, histogramas) para mostrarlas aquí.

**Entradas:** runs ya terminados, `WANDB_ENTITY` si la cuenta no se deduce sola, y la API key con la que hicisteis login.

**Salidas:** la tabla de `val/loss`, `train/loss`, `test/recon_error` (y KL si el run es un VAE) y las figuras del proyecto. No se recalcula un forward.

**No debe tocar otra persona:** no generéis estas figuras en local para “rellenar” la celda. Si falta un run, se entrena en la sección 9 o 10; no se simula.

### Preguntas guía

1. Para cada pérdida, ¿qué arquitectura abre más hueco entre el error de las buenas y el de las defectuosas? ¿Esa es la que tiene mejor `val/loss`? La validación no ve defectos, así que puede no coincidir.
2. ¿L2 difumina, L1 aplana color y SSIM conserva estructura pero también el defecto, en las dos arquitecturas o solo en la U-Net?
3. ¿`ssim_l1` (alpha 0.5) hereda lo peor o lo mejor de cada término? Mirad bordes en cable y textura en screw.
4. ¿Hay alguna subclase donde las dos arquitecturas fallan igual (defecto minúsculo, o un defecto que parece textura normal)?
5. Si tuvierais que desplegar un único par (arquitectura, pérdida) para marcar anomalías en estas cuatro clases, ¿cuál y por qué? La respuesta va en las conclusiones.


In [ ]:
from __future__ import annotations

if not RUN_COMPARISON:
    print("RUN_COMPARISON=False: no se consulta la API de WandB.")
else:
    tabla_final = comparar_runs(("vae", "u-net"))
    print(tabla_final)


## 12. Conclusiones

**Dueño:** todos. Sustituid los huecos cuando los 8 runs y la sección 11 estén vistos. No hace falta código nuevo.

### Par elegido

- Arquitectura:
- Pérdida:
- Por qué (hueco good/defectuoso, no solo `val/loss`):

### Qué se observa en imágenes buenas

- L1:
- L2:
- SSIM:
- SSIM+L1:

### Qué se observa en imágenes defectuosas

- VAE:
- U-Net (skips):
- Subclases que nadie separa bien:

### Limitaciones

- Un solo modelo para cuatro clases, resolución 128, `beta` y `alpha` fijos, val solo con buenas, sin máscaras de `ground_truth`.
- (añadid lo que hayáis visto en los runs)

### Trabajo que quedaría fuera de esta entrega

- (pendiente)
